# 分析の手習い — 演習用

架空のHPLC・生薬分析データで学ぶPython。サイトと同じ順序・演習IDです。各章の準備セルから始められます。初めての方はPREP-01から読みましょう。

測定値は学習用の式で生成したものです。実際の分析法、成分の効能、規格適合を判断する教材ではありません。

## 目次

- 最小限の準備：PREP-01
- pandas：PD-01 / PD-02 / PD-03 / PD-04 / PD-05 / PD-06 / PD-07 / PD-08 / PD-09 / PD-10 / PD-11 / PD-12 / PD-13 / PD-14 / PD-15 / PD-16 / PD-17 / PD-18 / PD-19 / PD-20 / PD-21 / PD-22 / PD-23 / PD-24 / PD-25 / PD-26
- NumPy：NP-01 / NP-02 / NP-03
- Matplotlib：MPL-01 / MPL-02 / MPL-03
- seaborn：SNS-01 / SNS-02 / SNS-03
- scikit-learn：ML-01 / ML-02 / ML-03 / ML-04 / ML-05 / ML-06
- 総合演習：ALL-01 / ALL-02 / ALL-03

## 0. 最小限の準備

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from IPython.display import display
from learning import check_answer

### PREP-01　セルを動かして、保存する

目安：20分 / 前提：Pythonで変数に値を入れた経験が少しある

#### 学習目標

- JupyterLabでNotebookを開く
- 実行順と保存・再起動の違いを説明する

まずサイトの「環境構築とJupyterLab」またはREADMEに従って、教材フォルダでJupyterLabを起動します。初回だけPythonライブラリを導入すれば、以降はオフラインで使えます。左のファイル一覧から `exercises.ipynb` を開きましょう。

Notebookは文章やコードを「セル」という小さな区画に分けたファイルです。コードセルを選んで **Shift+Enter** を押すと、Pythonを実行する仕組みであるカーネルへコードを送ります。`=` は右の値に名前を付け、`print()` は値を表示します。例題を実行し、120を180に変えてもう一度動かしてください。

保存はCmd+S / Ctrl+S。保存したコードと、カーネルが覚えている変数は別です。Kernelメニューで再起動すると変数は消えます。画面の上にあるセルが先に実行されるとは限りません。実行番号を確認し、章の準備セルから下へ順番に実行します。「Restart Kernel and Run All Cells」で先頭から再現できるかも確認できます。未回答の演習は `None`（まだ値がない印）のままで構いません。

#### 例題（演習とは条件が異なります）

In [ ]:
seconds = 120
minutes = seconds / 60
print(minutes)

120秒は2分です。再起動した後に計算セルだけを実行すると、secondsが未定義になる理由を考えましょう。

#### よくある間違い

- 保存すれば変数も復元すると思う
- 下のセルを先に動かし、古い変数を使う

### 演習 PREP-01

90秒を分に直し、数値をprep_01に入れて表示してください。求める単位は分です。

使用データ：この演習はCSVを使いません。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

1分は60秒です。

</details>

<details><summary>ヒント2</summary>

秒の値を60で割ります。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を prep_01 に入れてください。
prep_01 = None

In [ ]:
from learning import check_answer
check_answer('PREP-01', globals().get('prep_01'))

#### 自己確認と振り返り

- 保存と再起動で何が残り、何が消えるか
- 例題の値を変え、実行結果が変わったか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

## 1. pandas

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)


### PD-01　CSVを開き、生薬別に集計する

目安：25分 / 前提：PREP-01のセル操作

使う場面：装置から出力されたCSVを開き、対象の生薬を選び、平均成分量を確認したいときに使います。

扱う機能：`read_csv`, `DataFrame`, `Series`, `head`, `groupby`, `mean`

#### 学習目標

- 先頭5行を確認する
- 特定の生薬を抽出する
- 生薬別の平均成分量を求める

#### CSVを読み、表の先頭を確かめる
HPLCは、試料に含まれる成分を分けて検出する分析法です。この教材では、独立した検体の測定を1行にした架空CSVを使います。生薬名はherb、成分名はanalyte、成分量はcontent_mg_g列に入っています。成分量の単位はmg/gです。

`import pandas as pd`でpandasを読み込み、短い名前pdで呼べるようにします。`df = pd.read_csv('data/hplc.csv')`はファイルを読み、表をdfという名前で扱う指定です。引用符で囲んだ部分は文字列で、ファイルの場所や列名を表します。列名と行ラベルを持つ表をDataFrame、その1列を取り出したものをSeriesと呼びます。

`head(5)`は先頭5行を返します。例題の`display()`は、その結果をNotebookで見やすい表として表示します。列名や単位、各行が何の測定かを確かめてください。表示は5行でも、元のdfには360行が残っています。

#### 条件に合う行を取り出す
`df['herb']`で生薬名の列を選びます。続く`== 'ショウキョウ'`は、各行の値がショウキョウと等しいかを調べる比較です。一致すればTrue、一致しなければFalseになります。この真偽値の列を外側の角括弧に入れた`df[df['herb'] == 'ショウキョウ']`は、Trueの行だけを残します。

値を変数に入れる`=`と、等しいか調べる`==`を区別しましょう。抽出後の行ラベルに飛び番があっても、元の行の名前を保っているためです。抽出結果にhead(5)を付けるのは表示を短くするためで、対象が5件しかないという意味ではありません。

#### 生薬別の平均を読み、辞書で結果をまとめる
`df.groupby('herb')['content_mg_g'].mean()`は、生薬ごとに行をまとめ、成分量の平均を求めます。ドットで処理をつなぎ、丸括弧を付けて実行しています。結果のSeriesでは生薬名が行ラベル、平均が値です。平均では空欄に当たる欠損値が除かれます。別の生薬では別の成分を測っているので、mg/gの大小から薬効の優劣は判断できません。

演習では、表と平均を辞書にまとめます。辞書は名前であるキーと値の組です。例えば`record = {'name': 'ショウキョウ', 'count': 3}`なら、`record['name']`でショウキョウを取り出せます。値には数値だけでなくDataFrameも入れられるので、異なる種類の結果を意味の分かる名前で管理できます。

#### 例題（演習とは条件が異なります）

In [ ]:
import pandas as pd
df = pd.read_csv('data/hplc.csv')
display(df.head(5))
display(df[df['herb'] == 'ショウキョウ'].head(5))
display(df.groupby('herb')['content_mg_g'].mean())

最初は全体の先頭5行、次はショウキョウを抽出した表の先頭5行です。抽出後の行ラベルに飛び番があっても欠損ではありません。最後のSeriesは生薬名が行ラベル、平均成分量mg/gが値です。各平均は記録された値だけで計算されます。

#### コードを変えて試す

- 例題のhead(5)をhead(2)に変え、表示行数と元のdfの行数を比べてください。len(df)で行数を確認できます。
- 例題の生薬名をオウゴンに変えてください。抽出条件を変えても、生薬別集計の結果が変わらない理由を説明してください。

#### よくある間違い

- 比較に=を使う。=は代入、等しいかを調べるのは==です。
- head(5)で確認した5行だけを平均し、対象全件の平均と取り違える。
- herbの値や列名を引用符で囲まず、変数の名前として扱ってしまう。

### 演習 PD-01

カンゾウだけを抽出し、その全行と平均成分量mg/gをpd_01に辞書でまとめてください。辞書は名前と値の組を{キー: 値}で保持します。キーはrows（抽出表）とmean_content（平均値）とします。抽出表は120行、herbがすべてカンゾウであることを確認してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

まず条件に合う表を別の変数に置きます。

</details>

<details><summary>ヒント2</summary>

平均は抽出した表のcontent_mg_g列にmean()を使います。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_01 に入れてください。
pd_01 = None

In [ ]:
from learning import check_answer
check_answer('PD-01', globals().get('pd_01'))

#### 自己確認と振り返り

- head(5)は何を省略して表示しているか、元のdfと区別して説明できるか。
- カンゾウの全120行を取り出せたか。平均の単位と欠損の扱いを説明できるか。
- 辞書のrowsとmean_contentには、それぞれどの種類の値が入っているか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3.3：read_csv](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.read_csv.html)
- [pandas 2.3.3：SeriesとDataFrame](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/dsintro.html)

### PD-02　表を点検し、行と列を選ぶ

目安：30分 / 前提：PD-01

使う場面：分析前に列の型や空欄を点検し、必要な行と列を選んで大きい値から確認したいときに使います。

扱う機能：`info`, `describe`, `loc`, `iloc`, `sort_values`

#### 学習目標

- infoとdescribeを読み取る
- locとilocで選択する
- 値で並べ替える

#### 件数・型・ばらつきを確かめる
CSVが読めたら、想定どおりの表か点検します。`df.info()`は列名、非欠損件数、データ型を表示します。非欠損件数は値が記録されている行の数です。型は値の扱い方を表し、float64は小数を持つ数値、objectはこの表では主に文字列です。計算したい列がobjectなら、文字が混ざっていないか確かめます。

`describe()`は数値列の要約です。countは非欠損件数、meanは平均、stdはばらつきを表す標準偏差、minとmaxは最小値と最大値です。25%、50%、75%は小さい順に並べた値の位置を表し、50%が中央値です。四分位点とは、データを4等分する境界の値です。

例題では流速と保持時間の列を点検します。平均だけでなく範囲も読み、単位や桁が想定どおりか考えてください。全体の要約には異なる生薬や測定条件が混ざっているため、そのまま個々の生薬の特徴とはみなせません。

#### 列名のリストで必要な項目を選ぶ
Pythonのリストは、複数の値を順番付きでまとめるものです。`['herb', 'flow']`は生薬名と流速の列名を並べたリストです。`df[['herb', 'flow']]`では、外側の角括弧が列を選ぶ操作、内側が列名のリストに当たります。返る表はDataFrameで、列は指定した順になります。

`df['herb']`は1列のSeriesです。`df[['herb']]`は1列でもDataFrameになります。括弧の数によって、渡すものが文字列1つかリストかが変わります。例題を動かす前に、どの列が何列返るか予想しましょう。

#### locとilocで選び、値の順に並べる
行ラベルは、各行を識別する見出しです。`loc[行のラベルまたは条件, 列名]`は名前や条件で選び、`iloc[行の位置, 列の位置]`は0から数えた位置で選びます。`iloc[0:3, 0:3]`は先頭3行と先頭3列です。コロンで範囲を示すスライスでは、ilocの終点3は含みません。locでラベルの範囲を指定した場合は終点も含みます。

`sort_values('retention_min', ascending=False)`は保持時間を大きい順に並べた表を返します。ascendingは昇順かどうかの指定で、Falseなら降順です。結果を別の変数に入れない限り、元のdfの順番は変わりません。並べ替え後も行ラベルは残るので、表示位置と区別します。演習の`>=`は「以上」の比較です。`flow >= 0.8`なら0.8も対象に含みます。

#### 例題（演習とは条件が異なります）

In [ ]:
df.info()
display(df[['flow','retention_min']].describe())
display(df.loc[df['herb']=='ショウキョウ', ['sample_id','flow']].head(3))
display(df.iloc[0:3, 0:3])
display(df.sort_values('retention_min', ascending=False).head(3))

infoの非欠損件数はtemperatureが342件、content_mg_gが336件です。describeの50%は中央値です。保持時間の降順では25.086分の行が先頭になります。並べ替え後も行ラベルは元の値を保つので、上から何番目かはilocで指定します。

#### コードを変えて試す

- iloc[0:3, 0:3]をiloc[1:4, 0:3]へ変え、最初と最後の検体番号を比べてください。
- sort_valuesのascendingをTrueに変えてください。行ラベルと保持時間の並び方は、それぞれどう変わりますか。

#### よくある間違い

- ilocに列名を渡す。ilocは0から数えた位置で指定します。
- sort_values()の戻り値を受け取らず、元の表が変わったと思う。
- describeの平均や最大値だけを見て、単位と対象件数を確認しない。

### 演習 PD-02

flowが0.8以上の行からsample_idとflowの2列を選び、flowを降順に並べた表をpd_02に入れてください。位置による選択の復習として、結果の先頭3行をilocで表示してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

loc[条件, 列名のリスト]で選びます。

</details>

<details><summary>ヒント2</summary>

sort_valuesのascending=Falseを指定します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_02 に入れてください。
pd_02 = None

In [ ]:
from learning import check_answer
check_answer('PD-02', globals().get('pd_02'))

#### 自己確認と振り返り

- infoの非欠損件数とdescribeのcountは、何を数えているか。
- locとilocは、それぞれどのような基準で行を指定するか。
- 並べ替えた表の行ラベルが連番でない理由を説明できるか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3.3：行と列の選択](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/indexing.html)
- [pandas 2.3.3：describe](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.describe.html)

### PD-03　列を追加し、欠損値を扱う

目安：30分 / 前提：PD-02

使う場面：保持時間の単位をそろえたり、必要な測定値が空欄の行を整理したりするときに使います。

扱う機能：`copy`, `isna`, `sum`, `dropna`, `fillna`, `median`

#### 学習目標

- 単位を変換した列を作る
- 欠損数を数える
- 補完と除外の違いを説明する

#### 元の表を残して単位をそろえる
保持時間を別の資料へ転記するときは、分と秒の違いを確認します。元の測定表を残して加工するため、`work = df.copy()`で作業用の表を作ります。`work = df`だけでは、同じ表に別の名前を付けることになるため、独立した表の複製にはなりません。

`work['retention_sec'] = work['retention_min'] * 60`は、保持時間の列全体に60を掛け、新しい列へ入れます。各行について同じ計算が行われるので、繰り返しを書く必要はありません。計算後は列名と単位を対応させ、元のdfには秒の列が増えていないことも確かめます。単位を換えても、測定そのものの精度は変わりません。

#### 欠損を数えて、除外する列を決める
このCSVの空欄は欠損値として読み込まれ、数値列ではNaNと表示されます。欠損は記録された値がない状態で、測定値の0とは区別します。値がない理由はCSVだけでは分からないので、実際の分析では元の記録も調べます。

`isna()`は欠損ならTrueを返します。真偽値をsum()で足すとTrueが1、Falseが0として数えられ、欠損数を求められます。例題の`print()`は、説明の文字と計算結果を文字として表示するPythonの関数です。温度の欠損18件と成分量の欠損24件を確認してください。

`dropna(subset=['content_mg_g'])`は、成分量が欠けた行だけを除きます。360行から24行が除かれ、336行が残ります。subsetを指定しないdropna()では、温度だけが欠けた行も除外対象です。何の計算にどの列が必要かを決めてから、除外条件を指定します。

#### 空欄を補った値の意味を考える
行を残したまま空欄を埋める操作を補完と呼びます。例題の`fillna(work['temperature'].median())`は、温度の欠損を中央値で補います。中央値は値を小さい順に並べた中央の値です。fillna()の結果を温度列へ代入し、変更を反映します。成分量の列は補っていないので、その空欄は残ります。

補完しても、本当の測定値が判明したわけではありません。ここでは操作の練習として中央値を使います。成分量の空欄を理由なく0で埋めると、平均の意味も変わります。機械学習では、評価用データの情報が学習へ混ざるデータ漏洩を防ぐため、先にデータを分割し、訓練データだけから補完値を求めます。その手順はML章で扱います。

#### 例題（演習とは条件が異なります）

In [ ]:
work = df.copy()
work['retention_sec'] = work['retention_min'] * 60
print('欠損数:', work[['temperature','content_mg_g']].isna().sum())
work['temperature'] = work['temperature'].fillna(work['temperature'].median())
display(work[['temperature','retention_sec']].head())
print('成分量が記録された行数:',len(df.dropna(subset=['content_mg_g'])))

元の表では温度が18件、成分量が24件欠けています。温度を中央値で補っても成分量の空欄は残ります。成分量だけを条件に除外すると336行です。秒表示は分表示の60倍ですが、換算によって測定の精度が上がるわけではありません。

#### コードを変えて試す

- workへretention_secを追加した直後に、dfとworkの列名を比べてください。元のdfに新しい列がないことを確認します。
- 例題で温度を補う前後のwork['temperature'].isna().sum()を表示してください。成分量の欠損数はどうなるか予想してから実行します。

#### よくある間違い

- 欠損を測定値ゼロと取り違える。CSVの空欄だけでは、値がない理由は分かりません。
- subsetを指定せずdropna()を使い、温度が空欄の行まで消してしまう。
- work = dfだけで独立した表を複製したと思う。作業用の表はcopy()で作ります。

### 演習 PD-03

成分量content_mg_gが欠損している行だけを除き、retention_sec列を追加した表をpd_03に入れてください。元のdfを変えず、336行を残し、秒はretention_minの60倍にしてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

dropnaのsubsetで成分量列だけを指定します。

</details>

<details><summary>ヒント2</summary>

copy()した表へ新しい列を追加します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_03 に入れてください。
pd_03 = None

In [ ]:
from learning import check_answer
check_answer('PD-03', globals().get('pd_03'))

#### 自己確認と振り返り

- 336行を残すために、どの列の欠損だけを除いたか。
- 補完と除外は、行数や値をそれぞれどう変えるか。
- 元のdfを保ったまま加工できたか。補完値を実測値と区別して説明できるか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3.3：dropna](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.dropna.html)
- [pandas 2.3.3：fillna](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.fillna.html)

### PD-04　グループ集計とマスタ結合

目安：35分 / 前提：PD-03

使う場面：生薬別の平均と件数をまとめたり、別表の成分類を測定表へ加えたりするときに使います。

扱う機能：`groupby`, `agg`, `count`, `size`, `merge`

#### 学習目標

- 件数と平均を一緒に集計する
- mergeの行数増加を検出する

#### 平均と、その平均に使った件数を並べる
生薬別の集計では、平均だけでなく何件の値から計算したかを確認します。`groupby('herb')`で生薬別に行を分け、`agg()`へ集計方法を渡します。aggは、集計対象と処理をまとめて指定するための操作です。

例題の`{'content_mg_g': ['count', 'mean']}`は辞書です。キーで対象列を指定し、値として集計方法のリストを渡します。結果では、成分量の列の下にcountとmeanが並びます。countは欠損を除いた件数です。欠損も含めた行数は`df.groupby('herb').size()`で数えます。

同じ120行の生薬でも、平均に使った成分量の件数は異なります。両方を表示すると、集計の対象件数が分かります。ただし、件数だけで平均の確かさを判断できるわけではなく、値のばらつきや測定条件も確認が必要です。

#### 共通の列を手がかりに、別表の分類を加える
分類や設定をまとめた参照用の表をマスタと呼びます。この教材のanalytesは、成分名ごとに成分類classと検出波長detection_nmを持つ3行の表です。章の準備セルでdata/analytes.csvから読み込んであります。測定表へ分類を加えると、成分類ごとの集計ができます。

`df.merge(analytes, on='analyte', how='left', validate='many_to_one')`で結合します。onは行同士の対応を探す列、つまり結合キーの指定です。ここでは両方の表のanalyteが等しい行を対応させます。how='left'は左側のdfを残す結合で、マスタと一致しない測定も消えず、追加列が欠損になります。

#### 行数と追加列の欠損を点検する
many_to_oneは、測定表には同じ成分が複数行あってよいが、マスタ側では各成分が1行に限られるという条件です。validateでこの条件を確かめます。同じキーがマスタに重複すると結合後の行が増えるため、条件違反はエラーになります。

validateだけでは、すべての測定に一致するマスタがあるかは分かりません。`len(df)`と`len(joined)`で結合前後の行数を比べ、`joined['class'].isna().sum()`で追加した分類の空欄を数えます。同梱マスタのclassには空欄がないため、結合後に空欄があれば未一致を疑えます。元のマスタにも空欄がある場合は、未一致と区別して調べます。

例題は360行のままで、分類の欠損は0件です。演習ではこの分類で保持時間を集計します。分類ごとの平均差があっても測定条件も異なるので、分類だけが違いの原因とは言えません。

#### 例題（演習とは条件が異なります）

In [ ]:
display(df.groupby('herb').agg({'content_mg_g':['count','mean']}))
joined = df.merge(analytes,on='analyte',how='left',validate='many_to_one')
print('結合前後:',len(df),len(joined))
print('マスタ未一致:', joined['class'].isna().sum())
display(joined[['analyte','class','detection_nm']].head())

成分量のcountはオウゴン113件、カンゾウ109件、ショウキョウ114件です。各生薬の行数120件とは区別します。結合前後は360行のままで、classの欠損は0件です。このマスタでは、測定表のすべての成分に分類が対応しています。

#### コードを変えて試す

- df.groupby('herb').size()を追加し、成分量のcountと比べてください。両者の差が何を表すか考えます。
- left結合をhow='inner'へ変えてください。この同梱マスタでは行数が変わらない理由を、成分名の対応から説明してください。

#### よくある間違い

- マスタの重複で増えた行を、もともとあった測定と数えてしまう。
- countを欠損込みの全件数と思う。全行を数えるときはsize()を使います。
- many_to_oneが通れば、すべての測定にマスタが一致したと判断する。

### 演習 PD-04

測定表へanalytesマスタをleft結合し、成分類classごとの保持時間retention_minの平均を求め、平均値の昇順のSeriesをpd_04に入れてください。結合にmany_to_oneの確認を付けてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

結合はanalyte列をキーにします。

</details>

<details><summary>ヒント2</summary>

結合した表でclassごとにまとめ、mean()の後にsort_values()を使います。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_04 に入れてください。
pd_04 = None

In [ ]:
from learning import check_answer
check_answer('PD-04', globals().get('pd_04'))

#### 自己確認と振り返り

- 行数の確認と追加列の欠損確認では、それぞれ何を調べているか。
- 平均に使った件数が少ないグループを、結果だけで同じ確かさとみなしていないか。
- 成分類ごとの保持時間の違いを、その分類だけの影響と説明していないか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3.3：グループ集計](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/groupby.html)
- [pandas 2.3.3：結合と検証](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/merging.html)

### PD-05　ピボット集計を保存して読み直す

目安：30分 / 前提：PD-04

使う場面：生薬と装置を組み合わせて比較できる表を作り、別の作業でも使えるCSVとして残したいときに使います。

扱う機能：`pivot_table`, `pivot`, `reset_index`, `to_csv`, `read_csv`

#### 学習目標

- 2つの分類で集計する
- CSVを保存し読み直す
- 集計結果の限界を書く

#### 生薬と装置の組み合わせを表にする
生薬ごとの値を装置別に並べると、2つの分類を同時に確認できます。行と列に分類を置き、交差するセルに集計値を入れた表をピボット表と呼びます。例題では、生薬3行と装置2列に平均保持時間を配置します。

`pivot_table()`のindexには行の分類herb、columnsには列の分類instrumentを指定します。valuesは集計対象のretention_min、aggfuncは集計方法のmeanです。同じ生薬と装置の組み合わせに属する測定をまとめて平均します。別の操作であるpivot()は集計を行わないため、同じ組み合わせが複数行あるこの測定表へそのまま使うとエラーになります。

同じ行のLC-AとLC-Bを見比べれば、同じ生薬の平均保持時間を装置別に読めます。平均は欠損を除いて求めるため、測定値を成分量へ変えたら、各セルの計算に使った件数も確認してください。

#### 生薬名を残し、CSVへ保存する
集計後の表では、herbが行ラベルとして使われています。`reset_index()`でherbを通常の列へ戻すと、生薬名と2つの装置が並ぶ3列の表になります。戻した結果を保存するため、例題では`pivot.reset_index().to_csv(...)`と処理をつなぎます。

`to_csv()`は既定で行ラベルも書き出します。ここでは生薬名を通常列へ移したので、index=Falseを指定し、余分な連番を書き出さないようにします。生薬名を行ラベルにしたままindex=Falseで保存すると、生薬名がCSVに残りません。

保存先のoutputsフォルダは章の準備セルで作ります。入力のdataとは分け、加工前のCSVを上書きせずに比較できるようにします。同じ出力名で実行すると、そのファイルは更新されます。

#### 読み直して列と値を確かめる
ファイルができたら、`pd.read_csv()`で読み直します。herb、LC-A、LC-Bの列があり、3行の集計値を読めることを確認してください。余分な連番がUnnamedという列名で現れたら、保存時にindex=Falseを指定したかを見直します。

CSVは表の値を文字で保存する形式です。pandasのデータ型や画面上の書式をそのまま保存するものではなく、読込時に型が判断されます。小数には読み書きによるごく小さい誤差が出る場合もあります。値だけでなく列名と型も確かめる習慣を付けましょう。

装置別の平均差は、測定条件の構成にも左右されます。この表だけで装置性能の優劣は決められません。演習のTODOセルの下のコメントでは、何を集計した表かと、差の原因までは特定できない理由を説明します。

#### 例題（演習とは条件が異なります）

In [ ]:
pivot = df.pivot_table(index='herb',columns='instrument',values='retention_min',aggfunc='mean')
display(pivot)
pivot.reset_index().to_csv('outputs/retention_by_instrument.csv',index=False)
display(pd.read_csv('outputs/retention_by_instrument.csv'))

表は生薬3行×装置2列です。同じ行のLC-AとLC-Bで平均を比べられます。読み直すとherbも列になり、全3列になります。平均差があっても測定条件の構成は同じとは限らず、装置の性能差とは断定できません。

#### コードを変えて試す

- 例題のaggfuncをcountに変え、各セルの平均が何件の測定に基づくか確認してください。成分量へ変えた場合、欠損が件数にどう影響するか考えます。
- pivot.reset_index()を表示し、herbが行ラベルから通常列へ移る様子を確認してください。保存前の列構成と読み直した表を比べます。

#### よくある間違い

- index=Falseで保存する前にreset_index()を忘れ、生薬名を書き出さない。
- pivot()に同じ生薬と装置の複数行を渡し、自動で平均されると思う。
- 読み直した表を点検せず、ファイルが作られたことだけで保存成功と判断する。

### 演習 PD-05

生薬×装置の成分量content_mg_gの平均をピボット表pd_05に入れ、outputs/content_by_instrument.csvへ生薬名を通常列として保存してください。読み直して表示し、成分量が異なる理由を装置だけに帰せないことをTODOセルの下のコメントに1文で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

valuesをcontent_mg_g、aggfuncをmeanにします。

</details>

<details><summary>ヒント2</summary>

reset_index()してからindex=Falseで保存します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_05 に入れてください。
pd_05 = None

In [ ]:
from learning import check_answer
check_answer('PD-05', globals().get('pd_05'))

#### 自己確認と振り返り

- herbとLC-A、LC-Bの3列を読み直せたか。余分な連番列が増えていないか。
- pivot_tableのvaluesとaggfuncを変えると、各セルの意味はどう変わるか。
- 装置別の平均差から、分かることとまだ分からないことを1文で説明できるか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3.3：pivot_table](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.pivot_table.html)
- [pandas 2.3.3：to_csv](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.to_csv.html)

### PD-06　表を作り、インデックスで対応づける

目安：35分 / 前提：PD-05までの表操作（章の準備セルから独立して実行可能）

使う場面：指定された検体一覧の順に、測定表を並べて渡したい。

扱う機能：`DataFrame`, `Series`, `shape`, `index`, `columns`, `set_index`, `reset_index`, `reindex`, `rename`, `drop`

#### 学習目標

- 辞書やSeriesから小さな表を作る
- 位置と行ラベルを区別し、ラベルで計算されることを確かめる
- set_index・reindex・reset_indexで行の対応を整える

#### CSV以外から小さな表を作る
測定条件の候補や確認用の値を数行だけ入力したいときは、CSVを用意せず表を作れます。`pd.DataFrame({'列名': [値, 値]})`には、列名をキー、同じ長さのリストを値にした辞書を渡します。`pd.Series([値, 値], index=[行ラベル, 行ラベル])`は1列の値と行ラベルを持ちます。辞書は「名前から値を取り出す」入れ物、リストは「順番に値を並べる」入れ物でした。例題の補正値は操作を試すための架空値で、実際の分析条件を推奨するものではありません。

`shape`は表の大きさを`(行数, 列数)`で返します。この組はタプルと呼ばれ、`shape[0]`で行数、`shape[1]`で列数を読めます。`index`は行ラベル、`columns`は列名です。初期の0、1、2という行ラベルは測定番号とは限りません。

#### 同じ行ラベルを合わせて計算する
Series同士の足し算は、上から同じ位置を足すのでなく、同じ行ラベルを対応させます。例題では右側のSeriesを逆順にしてあります。それでもS0001同士が計算されることを確認してください。片方にしかないラベルは相手の値がなく、結果が欠損になります。意図しない欠損が出たら、値だけでなく両側の`index`も見ます。

#### 測定IDを行ラベルにする
`set_index('sample_id')`はID列を行ラベルに移します。これで`loc['S0001']`のように名前で選べます。指定するIDに重複がないかは先に確認してください。`reindex([...])`は、指定したラベルの順番と集合に合わせて表を作り直します。存在しないIDの行も作られ、値が欠損になります。存在するIDだけを選ぶ`loc`とはこの点が異なります。

`reset_index()`でラベルを再び列へ戻せます。`drop=True`を付けると元のラベルを捨てるため、測定IDを残したい場面では付けません。列名を読みやすくするには`rename(columns={'元の名前': '新しい名前'})`、列を除くには`drop(columns=[...])`を使います。どちらも通常は結果を返すので、変数で受け取ります。名前を変えても単位や値自体は変わりません。

#### 例題（演習とは条件が異なります）

In [ ]:
conditions = pd.DataFrame({'sample_id': ['S0001', 'S0002'], 'flow': [0.6, 0.8]})
print('行数と列数:', conditions.shape)
print('行ラベル:', conditions.index.tolist())
base = pd.Series([10.0, 20.0], index=['S0001', 'S0002'], name='base')
adjustment = pd.Series([2.0, 1.0], index=['S0002', 'S0001'], name='adjustment')
display(base + adjustment)
indexed = df[['sample_id', 'herb', 'flow', 'retention_min']].set_index('sample_id')
selected = indexed.reindex(['S0002', 'S0001', 'S9999'])
selected = selected.rename(columns={'flow': 'flow_ml_min'}).drop(columns=['retention_min'])
display(selected.reset_index())

足し算はS0001が11、S0002が22です。元の並び順ではなくIDが一致した相手を足しています。最後の表には指定順の3行があり、CSVに存在しないS9999は生薬名と流速が欠損です。

#### コードを変えて試す

- adjustmentの並び順だけを入れ替え、足し算の結果が変わるか確かめてください。
- reindexをlocに変え、存在しないIDに対する動作の違いを確認してください。

#### よくある間違い

- 行ラベルを行の位置と同じものだと思い、並べ替え後も番号だけで判断する
- reset_index(drop=True)で、残したい測定IDまで捨ててしまう
- renameで流速の単位も変換されたと思う

### 演習 PD-06

dfからsample_id・herb・flow・retention_minを選び、sample_idを行ラベルにしてください。行をS0003、S0001、S9999の順にそろえ、flowをflow_ml_minに改名し、retention_minを除いてからIDを列に戻した表をpd_06に入れてください。列の順はsample_id、herb、flow_ml_min、形は3行3列とします。S9999の欠損は『測定値がゼロ』といえるか、1文で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

set_indexでsample_idを行ラベルにすると、そのIDをreindexへ渡せます。

</details>

<details><summary>ヒント2</summary>

renameのcolumnsには、変更前と変更後の名前を辞書で渡します。

</details>

<details><summary>ヒント3</summary>

最後にreset_index()でIDを列へ戻します。drop=Trueは付けません。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_06 に入れてください。
pd_06 = None

In [ ]:
from learning import check_answer
check_answer('PD-06', globals().get('pd_06'))

#### 自己確認と振り返り

- 順序の違うSeries同士はどの値を対応させるか
- reindexで存在しないIDを指定すると何が起こるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [データ構造の基礎](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/dsintro.html)
- [ラベル選択とreindex](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/indexing.html)

### PD-07　複数条件で絞り込み、安全に更新する

目安：40分 / 前提：PD-06までの表操作（章の準備セルから独立して実行可能）

使う場面：複数の測定条件で対象を選び、点検用のフラグを付けたい。

扱う機能：`isin`, `between`, `query`, `loc`, `where`, `mask`, `copy`, `Copy-on-Write`

#### 学習目標

- 複数の条件を組み合わせる
- queryとlocで同じ条件を表す
- loc・where・maskを目的に合わせて使う

#### 条件を組み合わせる
「LC-Aで、流速が範囲内の測定」のように、実務では条件を組み合わせます。Seriesの条件を結ぶ記号は、両方を満たす`&`、どちらかを満たす`|`、条件を反転する`~`です。各条件をかっこで囲み、`(条件1) & (条件2)`と書きます。Pythonで1つの真偽値を扱う`and`や`or`を列全体に使うと、どの行を判定するか決まらずエラーになります。

候補が複数あるときは`isin(['LC-A', 'LC-B'])`、範囲は`between(0.6, 0.9, inclusive='both')`で表せます。`both`は両端を含む指定です。抽出前に条件のTrue件数を`sum()`で数えると、条件を追加した際にどれだけ対象が減ったか分かります。`~条件`はTrueとFalseを反転しますが、欠損を含む列では欠損の扱いも確認します。

#### 文章に近い条件をqueryで書く
`query('flow >= @minimum')`は列名を文字列の式に書く選択方法です。`@minimum`のように@を付けると、Python側の変数を参照します。ここでは自分で書いた固定の式だけを使います。`query`は式を評価するため、外部から受け取った文字列をそのまま渡す用途には使いません。結果をlocでの抽出と見比べ、どちらでも条件の意味を説明できるようにします。

#### 行を減らす操作と、値を置き換える操作
条件抽出は行数を減らします。一方、`where(条件)`は条件がTrueの値を残して、それ以外を欠損にします。`mask(条件)`はTrueの値を欠損にする逆の操作です。置換値を指定すれば欠損以外にもできますが、実測値を書き換える根拠とは別問題です。例題のフラグは点検用で、品質の合否ではありません。

別の表を加工したい場合は`copy()`で複製してから、`work.loc[条件, '列名'] = 値`と1回の指定で更新します。`work[条件]['列名'] = 値`のように2段階で選んでから代入する方法は避けます。どの表へ反映されるかが分かりにくいためです。Copy-on-Writeは共有データを変更時に分離する仕組みです。本教材のpandas 2.3では既定で無効ですが、将来の設定でも読みやすいよう、元表と更新先を明確にした書き方にそろえます。

#### 例題（演習とは条件が異なります）

In [ ]:
condition = (df['instrument'].isin(['LC-A'])) & df['flow'].between(0.6, 0.9)
print('対象件数:', condition.sum())
display(df.loc[condition, ['sample_id', 'instrument', 'flow']].head())
minimum = 0.9
display(df.query('flow >= @minimum')[['sample_id', 'flow']].head())
work = df[['sample_id', 'flow']].copy()
work['review'] = '通常確認'
work.loc[work['flow'] > 0.9, 'review'] = '流速条件を確認'
work['in_range'] = work['flow'].where(work['flow'].between(0.6, 0.9))
work['masked_high'] = work['flow'].mask(work['flow'] > 0.9)
display(work.head())
display(df.loc[(df['flow'] < 0.6) | (df['flow'] > 0.9), ['sample_id', 'flow']].head(3))
print('範囲外の件数:', (~df['flow'].between(0.6, 0.9)).sum())

条件抽出は該当行だけですが、whereとmaskで作った列は360行のままです。流速0.97の行ではin_rangeとmasked_highが欠損となり、reviewには点検用の文字列が付きます。dfの元の流速は変更していません。

#### コードを変えて試す

- betweenのinclusiveを'neither'に変え、境界上の行が除かれるか確認してください。
- 同じ範囲条件をqueryで書き、選ばれたsample_idを比べてください。

#### よくある間違い

- Series条件にand・orを使う
- &で結ぶ個々の比較のかっこを省く
- 二重の角括弧による選択の後へ代入し、更新先をあいまいにする

### 演習 PD-07

herbがカンゾウまたはオウゴン、かつflowが0.65以上0.85以下の行を選んでください。sample_id、herb、flowの3列をcopyし、review列を文字列『対象』で追加した4列の表をpd_07に入れます。元の行順と行ラベルを保ち、dfにはreview列を追加しないでください。条件の両端を含めたか、対象の意味を1文で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

生薬の候補はisin、流速の範囲はbetweenで表せます。

</details>

<details><summary>ヒント2</summary>

2つの条件を&で結び、locで必要な3列を選んでからcopyします。

</details>

<details><summary>ヒント3</summary>

新しい表だけにreview列を追加します。対象という印は分析上の合格を意味しません。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_07 に入れてください。
pd_07 = None

In [ ]:
from learning import check_answer
check_answer('PD-07', globals().get('pd_07'))

#### 自己確認と振り返り

- 条件抽出とwhereは、行数の点でどう違うか
- 点検フラグだけで測定の良否を決められないのはなぜか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [条件選択と更新](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/indexing.html)
- [Copy-on-Write](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/copy_on_write.html)

### PD-08　データ型を整え、変換できない値を見つける

目安：35分 / 前提：PD-07までの表操作（章の準備セルから独立して実行可能）

使う場面：数値と文字が混ざった記録を、欠損の理由を失わずに計算できる形へ直したい。

扱う機能：`dtypes`, `to_numeric`, `astype`, `Int64`, `boolean`, `string`, `convert_dtypes`, `select_dtypes`

#### 学習目標

- 文字列から数値への変換失敗を見つける
- 欠損を持てる整数・真偽値の型を使う
- 型の自動推定と意味の判断を区別する

#### 見た目が数値でも、計算できるとは限らない
測定記録には数値と「確認中」のような文字が同じ列に入ることがあります。画面に2と表示されても、文字列の`'2'`と整数の`2`は異なる値です。列の型である`dtype`を確かめてから計算します。例題ではCSVを変更せず、測定番号に対応した注入回数という練習用の小表を作ります。この回数は元のHPLCデータに含まれない合成値です。

`pd.to_numeric(列, errors='coerce')`は数値へ変換し、変換できない文字を欠損にします。エラーで止まらず進められますが、不正な入力が消えて見えるため、元の文字列も残します。`元の列.notna() & 変換後.isna()`で、入力があったのに数値へ変換できなかった行を探せます。空欄と誤入力を区別するには、元データの記録方法も確かめる必要があります。

#### 欠損を持てる整数と真偽値
一般的なNumPyの整数型は欠損を保持できず、欠損が混ざると小数の型になる場合があります。pandasの`Int64`は、欠損を`pd.NA`として持てる整数型です。`astype('Int64')`で指定します。大文字Iの`Int64`と小文字iの`int64`を区別してください。回数に2.5のような値が入っている場合、整数型への変換はまず値の意味を確認してから行います。小数の測定値を勝手に丸めて整数にしません。

真偽値にも欠損を持てる`boolean`型があります。Trueは確認済み、Falseは未確認、`pd.NA`は記録がない、というように3つの状態を分けられます。欠損はFalseと同じとは限りません。文字列には`string`型を指定できます。`astype`は変換後の型を自分で決める操作です。

#### 型の推定は、意味の点検の代わりにならない
`convert_dtypes()`は、各列を欠損対応の型へ推定して変換します。ただし「001」という検体IDを計算用の整数にするべきかなど、列の意味までは判断できません。`select_dtypes(include='number')`は数値型の列だけを取り出す方法です。IDが誤って数値型になっていれば選択されるので、型と列の意味の両方を確認してください。型を整えたら`dtypes`と欠損件数を再点検し、変換前後で増えた欠損が説明できるかを確かめます。

#### 例題（演習とは条件が異なります）

In [ ]:
raw = pd.DataFrame({'sample_id': df['sample_id'].head(4).tolist(),
                    'injections_text': ['2', '3', '確認中', None]})
number = pd.to_numeric(raw['injections_text'], errors='coerce')
converted = raw.copy()
converted['injections'] = number.astype('Int64')
converted['reviewed'] = pd.Series([True, False, None, True], dtype='boolean')
converted['sample_id'] = converted['sample_id'].astype('string')
display(converted)
display(converted.dtypes)
print('入力があるのに数値化できない行:')
display(raw.loc[raw['injections_text'].notna() & number.isna()])
display(pd.DataFrame({'count': [1, None, 3], 'flag': [True, None, False]}).convert_dtypes().dtypes)
print('数値型の列:', converted.select_dtypes(include='number').columns.tolist())

確認中とNoneはいずれも数値列では<NA>ですが、元の列を見れば入力状態の違いが分かります。注入回数はInt64、確認状態はbooleanです。型の一覧と表を合わせて確認してください。

#### コードを変えて試す

- 注入回数の'2'を'2.5'に変え、整数型へ変換する前に何を確認すべきか考えてください。
- convert_dtypesの前後で、値と型のどちらが変わったか表示してください。

#### よくある間違い

- errors="coerce"で欠損に変えた値を、正常な空欄として見逃す
- Int64とint64を同じ型だと思う
- 検体IDの先頭ゼロを失う数値変換を行う

### 演習 PD-08

dfの先頭4行のsample_idを使い、練習用の注入回数字列['1', '2', '未入力', '4']を数値化してください。変換できない値を欠損にし、Int64型のinjections列とします。確認状態[True, None, False, True]はboolean型のreviewed列にし、sample_id・injections・reviewedの順で4行3列のpd_08を作ってください。sample_idは元のCSVの値と型を保ちます。『未入力』を0で埋めると、解釈がどう変わるか説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

注入回数はまずpd.Seriesで作り、to_numericのerrorsをcoerceにします。

</details>

<details><summary>ヒント2</summary>

結果にastype("Int64")を使い、整数と欠損を共存させます。

</details>

<details><summary>ヒント3</summary>

reviewedはpd.Series(..., dtype="boolean")で作れます。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_08 に入れてください。
pd_08 = None

In [ ]:
from learning import check_answer
check_answer('PD-08', globals().get('pd_08'))

#### 自己確認と振り返り

- 変換前の文字列を残すと何を点検できるか
- 未確認のFalseと記録のないNAは同じ意味か

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [欠損を持つ整数型](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/integer_na.html)
- [欠損を持つ真偽値型](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/boolean.html)
- [to_numeric](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.to_numeric.html)

### PD-09　文字列を整え、測定ラベルを分解する

目安：35分 / 前提：PD-08までの表操作（章の準備セルから独立して実行可能）

使う場面：手入力と装置出力で異なる検体ラベルを、照合できる形に整えたい。

扱う機能：`str.strip`, `str.upper`, `str.replace`, `str.contains`, `str.extract`, `str.split`

#### 学習目標

- 空白・大文字小文字・固定文字をそろえる
- 部分一致と正規表現の違いを理解する
- ラベルからIDと装置名を取り出す

#### 文字の揺れをそろえる
装置から出力したラベルに余分な空白や大文字・小文字の違いがあると、同じ検体でも照合できません。文字列のSeriesでは`.str`に続けて操作を書きます。例題と演習のラベルは文字列処理を試すための合成値で、元CSVの測定IDを書き換えるものではありません。

`.str.strip()`は先頭と末尾の空白を除き、`.str.upper()`は英字を大文字にします。途中の空白はstripで消えないため、必要なら`.str.replace(' ', '', regex=False)`で半角空白を置き換えます。`regex=False`は最初の引数を、そのままの文字として扱う指定です。小さな操作を左から順につなぐ書き方をメソッドチェーンと呼びます。途中で何が変わるか分からなくなったら、各段階を別の変数にして表示します。

#### 部分一致で対象を探す
`.str.contains('LC-A', regex=False, na=False)`は文字列にLC-Aが含まれるかを返します。`na=False`は元の値が欠損ならFalseにする指定です。これは「欠損の行は今回の検索対象に入れない」という選択であり、記録が正常だと確認できたわけではありません。列自体に欠損がある場合は、`isna()`の件数も別に数えます。

#### 決まった形式を正規表現で読み取る
正規表現は「どんな文字がどの順に並ぶか」を表す小さな規則です。`r'^(S\d{4})\|(LC-[AB])$'`なら、`^`と`$`は文字列の始まりと終わり、`\d{4}`は数字4桁、`[AB]`はAかBの1文字を意味します。縦棒は規則の中で別の意味を持つため、`\|`で文字そのものを表します。丸かっこは取り出したい部分です。先頭の`r`は、バックスラッシュを含む文字列をPythonで書きやすくする記法です。

`.str.extract(規則)`は、丸かっこの中に一致した文字を列として返し、一致しない行は欠損になります。単に区切りで分けるなら`.str.split('|', expand=True, regex=False)`が読みやすい場合もあります。ただしsplitは「Sと数字4桁」という形式までは検査しません。整形後に一致しなかった行を表示し、見た目をそろえる処理と、記録の形式を点検する処理を分けて確認します。

#### 例題（演習とは条件が異なります）

In [ ]:
labels = pd.Series([' s0001 | lc-b ', 'S0002|LC-A', None, '確認中'], dtype='string')
clean = labels.str.strip().str.upper().str.replace(' ', '', regex=False)
display(pd.DataFrame({'before': labels, 'after': clean}))
print('LC-Aを含むラベル:')
display(clean[clean.str.contains('LC-A', regex=False, na=False)])
parts = clean.str.extract(r'^(S\d{4})\|(LC-[AB])$')
parts.columns = ['sample_id', 'instrument']
display(parts)
display(clean.str.split('|', expand=True, regex=False))
print('形式に合わないか欠損の件数:', parts['sample_id'].isna().sum())

余分な半角空白と英字の大小がそろい、最初の2行からIDと装置名を取り出せます。Noneと確認中はどちらも抽出結果が欠損ですが、元のラベルでは原因が異なります。splitは確認中も文字として残すため、形式検査の代わりにはなりません。

#### コードを変えて試す

- ラベルにS001やLC-Cを追加し、どちらの部分が規則に一致しないか確かめてください。
- replaceの対象を半角空白から別の固定文字へ変え、意図した文字だけが変わるか見てください。

#### よくある間違い

- stripで文字列の途中の空白まで消えると思う
- 点や縦棒を正規表現の特別な記号と知らず使う
- 一致しない行を全部捨て、入力ミスの内容を見失う

### 演習 PD-09

合成ラベル[' s0004 | lc-b ', 'S0002|LC-A', None, 'bad']をdtype='string'のSeriesにしてください。前後の空白を除き、英字を大文字にそろえ、半角空白をすべて除きます。その結果をnormalized列にし、Sと数字4桁・縦棒・LC-AまたはLC-Bという形式に完全一致する行からsample_idとinstrumentを抽出してください。normalized・sample_id・instrumentの順の4行3列の表をpd_09に入れます。最後の2行の抽出結果が欠損になる理由の違いを説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

strip、upper、replaceを順に使います。欠損を文字列Noneに変換しないでください。

</details>

<details><summary>ヒント2</summary>

extractにはr'^(S\d{4})\|(LC-[AB])$'を使えます。

</details>

<details><summary>ヒント3</summary>

抽出結果の2列へ名前を付け、normalizedを含むDataFrameを作ります。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_09 に入れてください。
pd_09 = None

In [ ]:
from learning import check_answer
check_answer('PD-09', globals().get('pd_09'))

#### 自己確認と振り返り

- 固定文字の置換でregex=Falseを明示する理由は何か
- extractで欠損になった行を、元の列でどう調べるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [文字列データの操作](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/text.html)
- [str.extract](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.str.extract.html)

### PD-10　欠損と重複を、1行の意味から点検する

目安：40分 / 前提：PD-09までの表操作（章の準備セルから独立して実行可能）

使う場面：取り込み後の二重登録や欠測が、集計件数と平均へどう影響するかを点検したい。

扱う機能：`isna`, `notna`, `count`, `size`, `duplicated`, `drop_duplicates`, `dropna`, `fillna`, `ffill`, `bfill`

#### 学習目標

- 欠損数と非欠損数を区別して数える
- 重複と反復測定を区別する
- 除外・補完の根拠と処理前後の件数を説明する

#### 集計の前に1行が何を表すかを決める
同じ検体名が2回現れても、同じ行の二重登録か、意図した再測定かで扱いは変わります。「1行が何を表すか」をデータの粒度と呼びます。元CSVでは1行が独立した検体の1成分の測定で、sample_idは一意です。例題では同じ行を選び直して、二重登録がある小表を練習用に作ります。実際の測定の繰り返しを示すデータではありません。

`duplicated(subset=['sample_id'], keep=False)`は、指定した列が同じ行をすべてTrueにします。まずその行を表示し、測定値まで一致しているかを確かめます。`drop_duplicates(subset=['sample_id'], keep='first')`は同じIDの先頭だけを残します。最初を採用する方針に根拠がないと、後から訂正した値を失うことがあります。異なる測定値を持つ同じIDが見つかったら、削除前に記録の意味を確認します。

#### 欠損の件数を、分母と一緒に示す
`isna()`は欠損か、`notna()`は欠損でないかを調べます。Seriesの`size`は欠損を含む要素数、`count()`は欠損を除いた数です。DataFrame全体のsizeは行数ではなく「行数×列数」になる点に注意してください。行数は`len(表)`または`shape[0]`で求めます。成分量の平均に何件が使われたかを説明するときは、全件数と非欠損件数を併記します。

#### 除外と補完は、用途を決めてから行う
`dropna(subset=['content_mg_g'])`は成分量がない行を除きます。温度だけが欠けた行まで落とさないよう、対象列を指定します。`fillna()`は空欄を埋める操作です。中央値で補っても実測値が得られたことにはならず、ばらつきや他の列との関係が変わる可能性があります。合成した欠測以外について、元CSVから欠測理由は分かりません。

前の行の値で埋める`ffill()`と後の行の値で埋める`bfill()`は、行の並びに意味がある場合に限って検討します。元CSVの行順を時間順だと思って使ってはいけません。例題では値が継続すると仮定した別の練習列で、動作だけを確認します。機械学習に使う補完値は、分割後の訓練データから学ぶ必要があり、ML章のPipelineで扱います。

#### 例題（演習とは条件が異なります）

In [ ]:
audit = df.iloc[[0, 1, 2, 1, 3, 4]][['sample_id', 'herb', 'content_mg_g']].copy()
audit.loc[audit['sample_id'] == df.iloc[2]['sample_id'], 'content_mg_g'] = np.nan
print('重複したIDの行:')
display(audit.loc[audit.duplicated(subset=['sample_id'], keep=False)])
unique = audit.drop_duplicates(subset=['sample_id'], keep='first')
summary = pd.Series({'重複除去前': len(audit), '重複除去後': len(unique),
                     '成分量の非欠損': unique['content_mg_g'].count(),
                     '成分量の欠損': unique['content_mg_g'].isna().sum()})
display(summary)
display(unique.dropna(subset=['content_mg_g']))
practice = pd.Series([1.0, None, 3.0, None], name='動作確認だけの練習列')
display(pd.DataFrame({'元の値': practice, '前の値': practice.ffill(), '後の値': practice.bfill(), '固定値': practice.fillna(0)}))

二重に選んだS0002を1行にすると6行が5行になります。成分量を欠損にした行は、重複除去だけでは消えません。最後の練習列ではbfillで末尾の欠損を埋められず、後に参照値がないことが分かります。

#### コードを変えて試す

- drop_duplicatesのkeepをFalseに変えると、どのIDの行がすべて消えるか確認してください。
- dropnaの対象列を増やす前に、それぞれの欠損件数を比べてください。

#### よくある間違い

- 同じ検体IDを見ただけで反復測定まで削除する
- DataFrame.sizeを行数だと思う
- 対象列を指定せずdropnaし、必要な行も除いてしまう

### 演習 PD-10

dfの行位置[0, 1, 2, 2, 3, 4]をこの順に選び、sample_id・herb・content_mg_gの3列をcopyしてauditを作ります。S0001のcontent_mg_gだけを欠損にしてください。この練習では同じIDの行は同じ記録の二重登録と分かっているので、IDの先頭だけを残します。pd_10を辞書とし、キーuniqueに重複除去後の表、usableに成分量の欠損行を除いた表、missing_countにuniqueの成分量の欠損数を入れてください。行ラベルと元の順を保ちます。除外とゼロ補完の解釈の違いを1文で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

ilocへ位置のリストを渡すと、同じ行を2度選ぶこともできます。

</details>

<details><summary>ヒント2</summary>

S0001の値はlocでnp.nanにします。np.nanは数値の欠損を表します。

</details>

<details><summary>ヒント3</summary>

drop_duplicatesのsubsetをsample_id、dropnaのsubsetをcontent_mg_gにします。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_10 に入れてください。
pd_10 = None

In [ ]:
from learning import check_answer
check_answer('PD-10', globals().get('pd_10'))

#### 自己確認と振り返り

- 同じIDで測定値が違う行は、何を確認してから扱うか
- 成分量の非欠損件数は、どの集計の分母に関係するか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [欠損データの扱い](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/missing_data.html)
- [drop_duplicates](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.drop_duplicates.html)

### PD-11　カテゴリと数値の区間を扱う

目安：35分 / 前提：PD-10までの表操作（章の準備セルから独立して実行可能）

使う場面：装置別の件数を数えたり、数値を決めた範囲で区分して比較したい。

扱う機能：`CategoricalDtype`, `astype`, `cat.categories`, `cat.ordered`, `cut`, `qcut`, `value_counts`, `get_dummies`

#### 学習目標

- カテゴリの集合と順序を指定する
- cutとqcutの区切り方を比較する
- カテゴリの件数とダミー変数の意味を説明する

#### 種類を表す列に、数値の大小を持ち込まない
装置名や生薬名は種類を表すカテゴリです。LC-AとLC-Bに0と1を付けても、BがAより優れているという意味にはなりません。大小関係を持たない分類を名義尺度、低・中・高のように順序がある分類を順序尺度と呼びます。並べる順を決めることと、数値として差や比を計算できることも別です。

`pd.CategoricalDtype(categories=[...], ordered=False)`はカテゴリとして認める値を指定します。`astype(型)`で列へ適用します。`ordered=True`なら指定順による大小比較が可能になります。指定した集合にない値は欠損になるので、変換前後の欠損数を比べます。生薬名の表示順は指定しても、生薬の優劣という順序は付けません。

#### 数値を区間にまとめる
`pd.cut(数値列, bins=[境界...], labels=[名前...])`は、自分で決めた境界で数値を分けます。例題は流速を3つの区間に分けますが、学習用の区切りであり、分析法の推奨範囲ではありません。既定の区間は左端を含まず右端を含みます。`include_lowest=True`なら最初の下限も含みます。境界の外にある値は欠損になるため、区切る前に最小値と最大値を確認してください。

`pd.qcut(数値列, q=3)`は、おおむね同じ件数になるよう分位点で区切ります。同じ数値が何度もあると件数がぴったり等しくならず、境界が重複してエラーになる場合もあります。その際は区間数を減らすなど、目的に合わせて考え直します。固定した条件範囲で比較したいときと、分布を同程度の件数で眺めたいときで使い分けます。

#### 件数とカテゴリの有無を確かめる
`value_counts(sort=False, dropna=False)`は、値ごとの件数を数えます。sort=Falseでカテゴリ順を保ち、dropna=Falseで欠損も数えます。`pd.get_dummies(列, prefix='instrument', dtype=int)`は、カテゴリごとに該当なら1、それ以外なら0の列を作ります。これをダミー変数と呼びます。1行に複数の条件を持つという意味ではなく、どのカテゴリだったかを数値列で表しています。機械学習では訓練・テストで列の集合がずれないよう、ML章の前処理で扱います。

#### 例題（演習とは条件が異なります）

In [ ]:
instrument_type = pd.CategoricalDtype(categories=['LC-A', 'LC-B'], ordered=False)
instrument = df['instrument'].astype(instrument_type)
print('装置のカテゴリ:', instrument.cat.categories.tolist(), '順序の有無:', instrument.cat.ordered)
bands = pd.cut(df['flow'], bins=[0.0, 0.65, 0.85, 1.0], labels=['低', '中', '高'], include_lowest=True)
display(bands.value_counts(sort=False, dropna=False))
quantile_bands = pd.qcut(df['flow'], q=3)
display(quantile_bands.value_counts(sort=False))
display(pd.get_dummies(instrument, prefix='instrument', dtype=int).head())

cutの件数は指定した境界によって決まります。qcutは分位点で境界を選ぶため、cutとは異なる範囲になります。ダミー変数はこのデータでは各行の合計が1となり、装置が1つ選ばれていることを表します。

#### コードを変えて試す

- cutの境界を変え、分類件数と欠損件数の両方がどう変わるか確認してください。
- CategoricalDtypeのcategoriesからLC-Bを除き、変換前後の欠損を比較してください。

#### よくある間違い

- 名義カテゴリに付けた番号を、順序や距離だと解釈する
- cutの境界の外にある値が欠損になったことを見逃す
- qcutの区間は必ず同じ幅、同じ件数になると思う

### 演習 PD-11

flowを境界[0.0, 0.6, 0.8, 1.0]、ラベル['低', '中', '高']、include_lowest=Trueで区分してください。pd_11を辞書とし、bandsに区分のSeries、countsにカテゴリ順の件数(value_countsのsort=False)、instrumentにinstrument列のダミー変数を入れます。ダミー変数はprefix='instrument'、dtype=intとし、元の行順を保ちます。流速0.6と0.8はそれぞれどの区間に入るか、件数の偏りが何を示すか説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

cutは既定で右端を含むため、0.6は最初の区間、0.8は次の区間に入ります。

</details>

<details><summary>ヒント2</summary>

cutの結果を変数へ入れ、そのSeriesのvalue_countsを使います。

</details>

<details><summary>ヒント3</summary>

get_dummiesには元のinstrument列を渡します。2つの装置に対応した2列になります。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_11 に入れてください。
pd_11 = None

In [ ]:
from learning import check_answer
check_answer('PD-11', globals().get('pd_11'))

#### 自己確認と振り返り

- cutとqcutでは、何を自分で指定しているか
- ダミー変数の1と0から装置の性能を判断できるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [カテゴリデータ](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/categorical.html)
- [cut](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.cut.html)
- [qcut](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.qcut.html)

### PD-12　日時と経過時間を扱う

目安：40分 / 前提：PD-11までの表操作（章の準備セルから独立して実行可能）

使う場面：測定の日時をそろえ、日をまたぐ経過時間を正しく計算したい。

扱う機能：`to_datetime`, `NaT`, `dt.hour`, `dt.dayofweek`, `dt.strftime`, `Timedelta`, `dt.total_seconds`, `date_range`, `dt.tz_localize`, `dt.tz_convert`

#### 学習目標

- 文字列を日時へ変換し、失敗した値を点検する
- dtで日付の要素と時間差を取り出す
- 地域の時刻とUTCの関係を説明する

#### 日時は文字列から変換して使う
測定日時を並べたり、前回からの経過時間を調べたりするときは、文字列のままでは扱いにくくなります。元CSVに日時列はないため、このレッスンでは測定IDに練習用の日時を付けます。作った日時から実際の装置の稼働や劣化傾向は判断できません。

`pd.to_datetime(列, format='%Y-%m-%d %H:%M', errors='coerce')`は指定した形式で日時へ変換します。%Yは4桁の年、%mは月、%dは日、%Hは24時間表記の時、%Mは分です。小文字mと大文字Mは意味が違います。変換できない値は日時の欠損`NaT`になります。元の文字列を残し、`isna()`で変換失敗を確認します。文字列の日付が月/日か日/月かあいまいなら、形式を先に決めます。

#### 日時の一部と経過時間を取り出す
日時Seriesの`.dt`は、日時向けの操作を使う入口です。`.dt.hour`で時、`.dt.dayofweek`で曜日番号を取り出せます。曜日番号は月曜が0、日曜が6です。`.dt.strftime('%Y-%m-%d')`は表示用の文字列に変えます。文字列へ変える前の日時列を計算用に残します。

日時同士を引くと、長さを表す`Timedelta`になります。`.dt.total_seconds() / 60`で、日をまたいだ差も含めて経過分へ変換できます。`.dt.seconds`だけでは日数分を含まないため、総経過時間にはtotal_secondsを使います。`pd.Timedelta(30, unit='min')`は30分の長さ、`pd.date_range(開始, periods=件数, freq='h')`は1時間間隔の日時列です。頻度Dなら日単位、hなら時間単位です。

#### どの地域の時刻かを記録する
地域を持たない日時に`.dt.tz_localize('Asia/Tokyo')`を使うと、その値を日本時間として解釈します。同じ瞬間を協定世界時UTCで表示するには`.dt.tz_convert('UTC')`を使います。localizeは時刻の意味を指定し、convertは同じ瞬間の表示地域を変えます。日本時間の9時はUTCの0時です。海外データでは夏時間で存在しない時刻や重複する時刻もあるため、時差を単純に足し引きする方法に頼らず、地域情報を使います。比較する日時の地域の有無が混在していないかも確認してください。

#### 例題（演習とは条件が異なります）

In [ ]:
practice = df[['sample_id']].head(4).copy()
practice['measured_text'] = ['2026-04-01 09:00', '2026-04-01 10:00', '確認中', '2026-04-02 11:00']
practice['measured_at'] = pd.to_datetime(practice['measured_text'], format='%Y-%m-%d %H:%M', errors='coerce')
practice['hour'] = practice['measured_at'].dt.hour
practice['weekday'] = practice['measured_at'].dt.dayofweek
practice['elapsed_minutes'] = (practice['measured_at'] - practice['measured_at'].iloc[0]).dt.total_seconds() / 60
display(practice)
tokyo = practice['measured_at'].dt.tz_localize('Asia/Tokyo')
display(pd.DataFrame({'Tokyo': tokyo, 'UTC': tokyo.dt.tz_convert('UTC')}))
print('30分後:', practice['measured_at'].iloc[0] + pd.Timedelta(30, unit='min'))
print('練習用の等間隔日時:', pd.date_range('2026-04-01 09:00', periods=3, freq='h').tolist())
print('表示用日付:', practice['measured_at'].dt.strftime('%Y-%m-%d').tolist())

確認中はNaTとなり、時・曜日・経過分も欠損です。翌日11時までの差は26時間、1560分なので、日をまたぐ分も含まれています。TokyoとUTCの列は時計の表示が異なりますが、同じ瞬間を表します。

#### コードを変えて試す

- 最後の日時を2日後に変え、total_secondsで求めた分数を手計算と比べてください。
- UTCへ変換した2つの日時でも差を計算し、日本時間での差と比較してください。

#### よくある間違い

- %m（月）と%M（分）を取り違える
- dt.secondsだけで日をまたぐ総経過時間を計算する
- tz_localizeとtz_convertを同じ操作だと思う

### 演習 PD-12

dfの先頭4行のsample_idに、合成日時['2026-04-01 09:00', '2026-04-01 10:30', '入力なし', '2026-04-02 09:00']を対応させてください。変換失敗をNaTにし、日本時間として解釈したmeasured_at列、同じ瞬間のUTC表示measured_utc列、最初の日時からの総経過分elapsed_minutes列を作ります。sample_id、measured_at、measured_utc、elapsed_minutesの順の4行4列をpd_12に入れてください。24時間後の経過分と、表示地域を変えても時間差が変わらない理由を説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

to_datetimeのformatとerrorsを指定してから、dt.tz_localize("Asia/Tokyo")を使います。

</details>

<details><summary>ヒント2</summary>

UTCへの変換はdt.tz_convert("UTC")です。

</details>

<details><summary>ヒント3</summary>

measured_atから先頭の日時を引き、dt.total_seconds()/60で総経過分を求めます。

</details>

In [ ]:
# TODO：問題文の条件に沿って結果を pd_12 に入れてください。
pd_12 = None

In [ ]:
from learning import check_answer
check_answer('PD-12', globals().get('pd_12'))

#### 自己確認と振り返り

- 日時の欠損NaTは、元のどの文字列から生じたか
- 日本時間をUTCへ変換したとき、経過時間そのものは変わるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [日時データの操作](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/timeseries.html)
- [時間差の操作](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/timedeltas.html)

### PD-13　複数条件で集計し、グループの値を各行へ戻す

目安：40分 / 前提：PD-04のgroupby、PD-12までの型と分類の扱い

使う場面：報告書に生薬・装置別の平均と有効件数を載せ、各検体の値も集団平均と照らし合わせる。

扱う機能：`groupby`, `agg（名前付き集計）`, `size`, `count`, `transform`, `filter`, `observed`, `dropna`

#### 学習目標

- 複数キーの名前付き集計で行数と有効件数を分ける
- transformとfilterの出力の形を区別する
- 分類の欠損と未使用カテゴリの扱いを指定する

#### 平均と一緒に、何件を数えたかを残す

生薬と装置の組合せごとに、測定件数と平均成分量を報告する場面を考えます。`groupby(['herb', 'instrument'])`は2列を組として分類します。生薬名だけでまとめた表より、どの装置で何件測ったかを細かく確認できます。ただし、装置ごとの測定条件も違うため、平均差だけで装置の影響は決められません。例題は先頭30行で仕組みを確かめ、演習では全360行に広げます。

`agg(rows=('content_mg_g', 'size'), valid=('content_mg_g', 'count'))`のような書き方を「名前付き集計」と呼びます。左側が出力する列名、右側の丸括弧は「対象列と集計方法」をひと組にしたタプルです。`size`は欠損を含む行数、`count`は指定列の欠損を除いた件数です。差を取れば、その列の欠損数が分かります。`as_index=False`を指定すると、分類列も通常の列として残せます。

#### 集計表を作るか、各行に値を付けるか

`agg`はグループごとに結果をまとめます。一方、`transform('mean')`は、各行が属するグループの平均を元と同じ行数・行ラベルで返します。測定値からグループ平均を引けば、その検体が同じ生薬の平均からどれくらい離れるかを調べられます。これは偏差であり、単位は元の成分量と同じmg/gです。元の成分量が欠損している行の偏差も欠損になります。

`filter`はグループ全体を残すか除くかを判断します。`lambda group: len(group) >= 5`は「グループを受け取り、5行以上かを返す」短い関数です。`lambda 引数: 式`では、式の値がそのまま結果になります。例題では生薬と装置の組合せごとに行数を判定するため、条件を満たした組の行がすべて残ります。5という値は練習用で、分析結果を採用する基準ではありません。

#### 分類の欠損と、存在しない分類を区別する

`dropna=False`は分類キーが欠損した行もひとつの組として数えます。これは測定値の欠損を平均へ含める指定ではありません。`observed=True`は分類列がカテゴリ型のとき、実際に現れた分類の組だけを出します。登録済みだが未使用のカテゴリまで並べたい場合は`False`を選びます。後半の小さな練習表では、未使用のLC-Cを集計から除き、欠損した装置名は残しています。集計前に、欠損・未観測・測定値ゼロの意味を分けておきましょう。

#### 例題（演習とは条件が異なります）

In [ ]:
example_df = df.head(30).copy()
groups = example_df.groupby(['herb', 'instrument'], as_index=False, observed=True, dropna=False)
summary = groups.agg(rows=('content_mg_g', 'size'), valid=('content_mg_g', 'count'), mean_content=('content_mg_g', 'mean'))
display(summary)
work = example_df.copy()
work['group_mean'] = work.groupby('herb')['content_mg_g'].transform('mean')
work['deviation'] = work['content_mg_g'] - work['group_mean']
display(work[['herb', 'content_mg_g', 'group_mean', 'deviation']].head(5))
kept = example_df.groupby(['herb', 'instrument']).filter(lambda group: len(group) >= 5)
print('5行以上ある組に属する行数:', len(kept))
# 分類の欠損と未使用カテゴリを確かめるための小さな架空表
key_demo = pd.DataFrame({'instrument': pd.Categorical(['LC-A', None, 'LC-B'], categories=['LC-A', 'LC-B', 'LC-C']), 'value': [1, 2, 3]})
display(key_demo.groupby('instrument', dropna=False, observed=True).size())

例題の先頭30行から6組の集計ができます。rowsの合計は30、validは28で、成分量の欠損は2件です。transformで返る偏差は30要素のままで、同じ生薬のこの30行中の平均からの差を表します。filterで5行以上の組だけを残すと、6行・6行・7行の3組、計19行になります。演習では全360行を対象にするため、平均も件数も例題とは変わります。

#### コードを変えて試す

- transformの分類を生薬×装置へ変え、同じ行の偏差がどう変わるか確認してください。
- 例題のfilterのしきい値を5から4へ変え、残る組と行数を確認してください。

#### よくある間違い

- 分類列のdropnaと、平均する測定値の欠損除外を混同する
- transformで返る360行を、6行の集計表と同じものだと思う
- 欠損を含む行数を知りたいのにcountだけを見る

### 演習 PD-13

生薬×装置ごとに、rows（成分量の欠損を含む行数）、valid（成分量の非欠損件数）、mean_content（平均成分量）をこの順で持つ集計表を作り、herbとinstrumentを通常列として先頭に残してください。分類キーは昇順とします。さらに生薬ごとの平均成分量を各行へ戻し、元の成分量から引いたSeriesを作ってください。pd_13を、summaryに6行の集計表、deviationに元と同じ行ラベルの360要素のSeriesを入れた辞書にします。TODOセルの下のコメントには、rowsとvalidが違う理由と、欠損行の偏差が欠損になる理由を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

groupbyへ2列のリストを渡し、as_index=Falseと名前付きaggを使います。

</details>

<details><summary>ヒント2</summary>

グループ平均を各行へ戻すにはgroupbyの後にtransformを使います。

</details>

<details><summary>ヒント3</summary>

辞書は {名前: 値} で作ります。summaryとdeviationの行数は同じではありません。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_13 に入れてください。
pd_13 = None

In [ ]:
from learning import check_answer
check_answer('PD-13', globals().get('pd_13'))

#### 自己確認と振り返り

- agg・transform・filterはそれぞれ何行を返すか
- rowsとvalidの差が大きい組では何を確認するか
- 平均からの偏差を薬効や規格適否と取り違えていないか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：グループ集計](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/groupby.html)
- [pandas 2.3：transform](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.core.groupby.SeriesGroupBy.transform.html)

### PD-14　結合方法を選び、未一致と行数の増加を調べる

目安：40分 / 前提：PD-04のmerge、PD-13の複数キー集計

使う場面：成分マスタに未登録の測定があってもデータを消さず、登録不足を報告する。

扱う機能：`merge`, `how=inner/left/outer`, `validate`, `indicator`, `concat`, `ignore_index`, `join`

#### 学習目標

- inner・left・outer結合を目的で選ぶ
- validateとindicatorで結合結果を点検する
- concatとjoinの用途を説明する

#### どの行を残すかを先に決める

測定表と成分マスタを結合するときは、まず各表の1行が何を表すかを確かめます。これを表の「粒度」と呼びます。この教材では、測定表は1検体・1成分の測定、マスタは1成分につき1行です。測定表に同じanalyteが繰り返し現れるのは正常ですが、マスタの重複は1件の測定を複数行へ増やす原因になります。

`merge(..., on='analyte', how='left')`は左の測定をすべて残します。`inner`は両方にキーがある測定だけを残し、`outer`は左右どちらかにだけあるキーも残します。右だけにあるマスタ行には測定値が付かず欠損になります。単に列を追加するつもりでinnerを使うと、未登録の成分の測定が消えてしまうため、分析対象を絞る意図があるかを先に決めます。

#### 未登録と重複は別々に点検する

`validate='many_to_one'`は「左は同じキーが複数回あってよいが、右は一意」を確認します。左右とも一意なら`one_to_one`です。左右で同じキーがそれぞれ2行と3行あれば、結合後には組合せの6行ができます。この多対多結合が意図したものかを確認せず、そのまま測定件数を数えてはいけません。

`indicator=True`で付く`_merge`列は、`both`が両方で一致、`left_only`が左だけ、`right_only`が右だけという意味です。validateを通っても未登録キーは残るので、この列と結合前後の行数を確認します。例題ではマスタの一部を除き、「練習用成分」を1行追加した小さなマスタをあえて作ります。右側に付く値が欠損した理由を、元から欠損していた場合と結合で生じた場合に分けて考えましょう。なおpandasでは、欠損したキー同士も一致として結合されます。結合キーの欠損は事前に点検します。

#### 行を継ぎ足す場合と、行ラベルを使う場合

同じ列を持つ測定表を上下につなぐときは`pd.concat([表1, 表2], ignore_index=True)`を使います。角括弧内は、つなぐ表を順に並べたリストです。ignore_indexは連番を振り直す指定で、sample_idの重複を消す機能ではありません。`join`は行ラベルをキーに情報を付けるときに便利です。例題ではマスタを`set_index('analyte')`で成分名の行ラベルにし、測定表のanalyte列と対応させています。同じ内容でも、元データの持ち方に合う方法を選べます。

#### 例題（演習とは条件が異なります）

In [ ]:
joined = df.merge(analytes, on='analyte', how='left', validate='many_to_one', indicator=True)
print('通常マスタの照合:', joined['_merge'].value_counts().to_dict())
# 以下の欠落・追加は結合方法を比較するための練習用変更
extra = pd.DataFrame({'analyte': ['練習用成分'], 'class': ['練習'], 'detection_nm': [254]})
master_demo = pd.concat([analytes.iloc[:2], extra], ignore_index=True)
inner = df.merge(master_demo, on='analyte', how='inner', validate='many_to_one')
left = df.merge(master_demo, on='analyte', how='left', validate='many_to_one')
outer = df.merge(master_demo, on='analyte', how='outer', validate='many_to_one', indicator=True)
print('inner / left / outerの行数:', len(inner), len(left), len(outer))
display(outer['_merge'].value_counts())
combined = pd.concat([df.iloc[:3], df.iloc[3:6]], ignore_index=True)
display(combined[['sample_id', 'herb']])
by_index = df.head(3).join(analytes.set_index('analyte'), on='analyte', validate='many_to_one')
display(by_index[['sample_id', 'analyte', 'class']])

完全なマスタでは360行がbothです。練習用マスタのinnerは240行、leftは360行、outerは361行になります。outerの1行増加は右だけにある練習用成分で、測定が1件増えたわけではありません。concatの連番変更と、検体IDの一意性は別に確認します。

#### コードを変えて試す

- 演習のhowをinnerへ変え、件数と生薬の構成を確かめてください。
- 練習用マスタに同じanalyteの行を足した場合、many_to_oneがなぜエラーを返すかを説明してください。

#### よくある間違い

- left結合なら右キーが重複していても行数は増えないと思う
- validateを指定すれば未登録キーも検出できると思う
- concatのignore_indexで重複した測定が除去されると思う

### 演習 PD-14

練習として、analytesからanalyteがグリチルリチンの行だけを除いたマスタを作ってください。dfへanalyteをキーにleft結合し、many_to_oneの確認とindicator=Trueを指定した表をpd_14に入れます。元の測定を360行のまま保ち、末尾に_merge列を持たせてください。_mergeの件数を表示し、left_onlyの検体がどの生薬かを確認してください。TODOセルの下のコメントには、この状態でinner結合へ変えた場合に分析対象がどう変わるかを書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

マスタの抽出条件は analytes['analyte'] != 'グリチルリチン' です。

</details>

<details><summary>ヒント2</summary>

mergeのhow・validate・indicatorをそれぞれ指定します。

</details>

<details><summary>ヒント3</summary>

未一致行は _merge == 'left_only' で選べます。生薬名はunique()で重複を除いて確認できます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_14 に入れてください。
pd_14 = None

In [ ]:
from learning import check_answer
check_answer('PD-14', globals().get('pd_14'))

#### 自己確認と振り返り

- innerに変更するとどの測定が対象から外れるか
- left_onlyとright_onlyはそれぞれ何の不足を示すか
- 結合前後の行数だけでは見つからない問題は何か

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：結合・連結](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/merging.html)
- [pandas 2.3：merge](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.merge.html)

### PD-15　縦長・横長の表を変換し、集計の単位を守る

目安：40分 / 前提：PD-05のpivot_table、PD-14の結合

使う場面：測定項目が列に分かれた報告表を、可視化や項目別集計で使える縦長の表へ変える。

扱う機能：`melt`, `pivot`, `pivot_table`, `stack`, `unstack`, `explode`, `crosstab`

#### 学習目標

- meltで複数の測定列を縦長へ変える
- pivotとpivot_tableの前提を区別する
- stack・explode・crosstabの用途を説明する

#### 測定項目を列にするか、行にするか

1検体につき成分量・保持時間を別々の列に持つ表は、人が項目を見比べやすい形です。一方、項目名と値を2列へまとめた「縦長」の表は、項目ごとの図や集計を同じ手順で作りやすくなります。`melt`の`id_vars`には検体IDや生薬など残したい識別列を、`value_vars`には縦に並べたい測定列を指定します。`var_name`が項目名の列名、`value_name`が数値の列名です。

例題では4検体×2項目を8行へ変えます。測定が8件になったのではなく、元の4件を項目別に2行ずつ表した結果です。成分量mg/gと保持時間minを同じvalue列へ置くため、measurementで区別して使います。value列全体の平均には意味がありません。異なる単位の値を同じ軸の図で比べることも避けます。CSV内の欠損は、形を変えても欠損として残ります。

#### 横長へ戻すときは、重複の意味を決める

`pivot(index='sample_id', columns='measurement', values='value')`は、検体IDと項目名の組が一意である場合に横長へ戻します。同じ検体・項目に複数行あれば、どの値を置くか決められないためエラーになります。重複が再測定などの意味を持ち、平均へまとめる判断をした場合は`pivot_table(..., aggfunc='mean')`を使います。エラーを消す目的だけで平均してはいけません。

`stack`は列のラベルを行側へ移し、`unstack`は行ラベルの階層を列側へ移します。例題では生薬×装置のピボット表を`stack(future_stack=True)`で縦へ移します。ここで生薬と装置の2段の行ラベルができ、次のレッスンで扱うMultiIndexになります。教材のpandas 2.3ではfuture_stack=Trueで新しい実装を明示します。stackは通常の1段の列でも使えます。

#### セル内のリストと、分類ごとの件数

`explode`は1つのセル内にあるリストを要素ごとの行へ展開します。例題のタグ表は処理を試すために作った架空のメモです。角括弧で囲んだ`['再確認', '温度']`は2つの文字列を持つリストなので、展開すると2行になり、sample_idは両方の行に付きます。カンマを含む普通の文字列を渡すだけでは分割されません。その場合は文字列処理のstr.splitなどで先にリストへ変えます。

`crosstab(df['herb'], df['instrument'])`は、生薬×装置ごとの件数表を返します。平均の表と併せて読むと、何件から平均を求めたかを確認できます。ただしこの件数は成分量の欠損も数えるため、平均に使った有効件数を知りたい場合はPD-13のcountを使います。

#### 例題（演習とは条件が異なります）

In [ ]:
small = df[['sample_id', 'herb', 'content_mg_g', 'retention_min']].head(4)
long = small.melt(id_vars=['sample_id', 'herb'], value_vars=['content_mg_g', 'retention_min'], var_name='measurement', value_name='value')
display(long)
wide = long.pivot(index='sample_id', columns='measurement', values='value')
display(wide)
means = df.pivot_table(index='herb', columns='instrument', values='retention_min', aggfunc='mean')
stacked = means.stack(future_stack=True)
display(stacked)
print('unstackで元の表へ戻る:', stacked.unstack().equals(means))
# タグはexplodeを試すための架空のメモ
notes = pd.DataFrame({'sample_id': ['S0001', 'S0002'], 'tags': [['再確認', '温度'], ['流速']]})
display(notes.explode('tags', ignore_index=True))
display(pd.crosstab(df['herb'], df['instrument']))

meltの出力は成分量4行、保持時間4行の計8行です。pivotで戻した表は検体IDの行ラベルを持つ4行になります。タグ表の3行は3件の測定ではありません。crosstabは6つの組の測定件数を示しますが、欠損を除いた成分量の件数とは区別します。

#### コードを変えて試す

- meltのvalue_varsを1列だけにして、行数がいくつになるか確かめてください。
- crosstabの出力とPD-13のvalidを比較し、件数が異なる理由を説明してください。

#### よくある間違い

- melt後の行数を測定した検体数として報告する
- 単位の違うvalueを区別せず平均する
- pivotが失敗した理由を調べずpivot_tableで平均してしまう

### 演習 PD-15

df全360行について、sample_idとherbを識別列として残し、content_mg_gとretention_minをこの順で縦長に変換してください。項目名の列をmeasurement、値の列をvalueとし、列順がsample_id・herb・measurement・valueの720行の表をpd_15に入れます。元の欠損24件は削除しません。sample_idとmeasurementの組が重複しないことを確認し、TODOセルの下のコメントにvalue列全体の平均を求めない理由を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

meltのid_varsとvalue_varsには、それぞれ列名のリストを渡します。

</details>

<details><summary>ヒント2</summary>

var_nameとvalue_nameで新しい2列の名前を決めます。

</details>

<details><summary>ヒント3</summary>

duplicated(subset=['sample_id', 'measurement']).sum()で組の重複を数えられます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_15 に入れてください。
pd_15 = None

In [ ]:
from learning import check_answer
check_answer('PD-15', globals().get('pd_15'))

#### 自己確認と振り返り

- melt後に何をキーとして元の1件を識別するか
- 縦長から横長へ戻せないとき、どの重複を確かめるか
- explode前後で測定そのものの件数は増えたか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：表の形の変換](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/reshaping.html)
- [pandas 2.3：melt](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.melt.html)
- [pandas 2.3：stack](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.stack.html)

### PD-16　複数の行ラベルを使い、表の並びをそろえる

目安：35分 / 前提：PD-02のloc、PD-15のstackとunstack

使う場面：報告書の分類順に集計を並べ、まだ測定していない装置も欠損として表示する。

扱う機能：`MultiIndex`, `set_index`, `sort_index`, `xs`, `loc`, `reset_index`, `reindex`, `Series.add`, `fill_value`

#### 学習目標

- MultiIndexで生薬・装置の2段の行ラベルを扱う
- reindexで必要な分類を明示的に並べる
- ラベルをそろえた算術と欠損の意味を説明する

#### 分類が2つあれば、行ラベルも2段にできる

生薬と装置で集計した表では、「カンゾウかつLC-A」のような組が1行を特定します。`set_index(['herb', 'instrument'])`で2つの列を行ラベルにすると、MultiIndex（複数の階層を持つ行ラベル）になります。元の測定表にも設定できますが、同じ組の測定が多数あるため一意ではありません。例題では平均流速を先に集計し、1組が1行の表を作ります。flow_meanの単位はmL/minです。演習では同じ操作を平均保持時間に使います。

`sort_index()`はラベルで並べ替える操作です。複数階層で範囲を指定するときは、ラベルを並べておくと扱いやすくなります。`loc[('カンゾウ', 'LC-A'), :]`の丸括弧は、外側の生薬と内側の装置を順番に指定するタプルです。`xs('LC-A', level='instrument')`は、全生薬からLC-Aだけを取り出します。xsは指定した階層を既定で取り除くので、結果の行ラベルは生薬だけになります。

#### 必要な行を、必要な順に置く

報告様式に載せたい行の順番が決まっているときは`reindex`を使います。`pd.MultiIndex.from_product([生薬のリスト, 装置のリスト], names=['herb', 'instrument'])`は、2つのリストのすべての組合せを行ラベルにします。これをreindexへ渡すと、元にない組合せも行として現れ、その値は欠損になります。例題のLC-Cは練習用に要求した未観測の装置で、ゼロを測定したわけではありません。

行ラベルをCSVへ出す場合は、`reset_index()`で生薬・装置を通常列へ戻します。行ラベルが何段か、列名がどこにあるかを確かめると、保存後に識別情報を失う誤りを防げます。reindexは並べ替えだけでなく、ラベルの追加や除外も行うため、前後の行数も確認します。

#### 計算は、行の位置よりラベルが優先される

Series同士の加算は、同じ位置の値を足す操作ではありません。片方の順序が違っても、pandasは同じラベル同士を対応させます。例題の架空の予定件数では、LC-AとLC-Bの順序を逆にした追加件数を正しく対応させています。ラベルが片方にしかない場合は、そのまま足すと欠損になります。

`.add(相手, fill_value=0)`は片方にない値などを0として計算する指定です。これは「その装置の予定件数は記録がなければ0件」という意味が確かなときに使えます。測定値が未記録だからという理由で0を補ってはいけません。両方とも欠損なら、この指定でも結果は欠損です。意図せず欠損が現れたときは、値を埋める前にラベルの誤記や欠落を点検しましょう。

#### 例題（演習とは条件が異なります）

In [ ]:
summary = df.groupby(['herb', 'instrument'], as_index=False).agg(flow_mean=('flow', 'mean'))
indexed = summary.set_index(['herb', 'instrument']).sort_index()
display(indexed)
display(indexed.loc[('カンゾウ', 'LC-A'), :])
display(indexed.xs('LC-A', level='instrument'))
order = pd.MultiIndex.from_product([['カンゾウ', 'ショウキョウ', 'オウゴン'], ['LC-A', 'LC-B', 'LC-C']], names=['herb', 'instrument'])
display(indexed.reindex(order).reset_index())
# 以下は、記載がなければ0件と決めた架空の予定件数
planned = pd.Series([3, 4], index=['LC-A', 'LC-B'])
added = pd.Series([2, 1], index=['LC-B', 'LC-A'])
display(planned + added)
extra = pd.Series([2, 1], index=['LC-B', 'LC-C'])
display(planned.add(extra, fill_value=0))

例題のflow_meanは平均流速（mL/min）です。xsの結果は生薬3行で、reindex後は3生薬×3装置の9行となり、LC-Cの平均流速はすべて欠損です。別の練習である予定件数は、順番を逆にしたSeries同士でもLC-Aが4件、LC-Bが6件になります。行の位置で足した結果とは異なります。

#### コードを変えて試す

- xsのlevelをherb、ラベルをカンゾウに変え、結果の行ラベルを確認してください。
- 予定件数の例でextraをそのまま加算した結果と、fill_value=0付きの結果を比較してください。

#### よくある間違い

- locで外側と内側のラベルの順序を逆にする
- 未観測のLC-Cの平均を0として埋める
- Series同士の演算を配列のような位置同士の計算と思う

### 演習 PD-16

生薬×装置ごとのretention_minの平均を、列がretention_minだけの表にしてください。行はMultiIndexとし、外側のherbはカンゾウ・ショウキョウ・オウゴンの順、各生薬の内側のinstrumentはLC-A・LC-B・LC-Cの順にそろえ、9行の表をpd_16に入れます。元にないLC-Cの3行は欠損のまま残してください。TODOセルの下のコメントには、この欠損を0分で埋めない理由を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

複数キーでgroupbyし、[['retention_min']]を選んでmean()を使うと1列のDataFrameになります。

</details>

<details><summary>ヒント2</summary>

MultiIndex.from_productで必要な行ラベルの全組合せを作ります。

</details>

<details><summary>ヒント3</summary>

平均の表にreindexを使います。存在しない組合せは自動的に欠損になります。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_16 に入れてください。
pd_16 = None

In [ ]:
from learning import check_answer
check_answer('PD-16', globals().get('pd_16'))

#### 自己確認と振り返り

- xsで選んだ階層が結果のラベルから消えることを確認したか
- 欠損は元の測定値由来か、reindexで作った行由来か
- 計算前に両方のラベルを点検できるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：MultiIndex](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/advanced.html)
- [pandas 2.3：reindex](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.reindex.html)
- [pandas 2.3：ラベルをそろえた加算](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.add.html)

### PD-17　合成日時で時間間隔ごとの集計を練習する

目安：35分 / 前提：PD-12の日時の基礎、PD-13の名前付き集計

使う場面：日時を持つデータの集計方法を、実測時刻と区別した合成日時で練習する。

扱う機能：`date_range`, `DatetimeIndex`, `to_datetime`, `resample`, `asfreq`, `closed`, `label`

#### 学習目標

- DatetimeIndexと時間間隔を設定する
- resampleで期間ごとの平均と有効件数を求める
- asfreqが作る欠損と、実測の欠損を区別する

#### 日時がある場合に使う操作を、練習用の日時で試す

時系列は、日時に沿って並ぶデータです。実際の測定日時があれば、日別の件数や時間帯ごとの平均を求められます。ただし、この教材のCSVに日時の列はなく、行順にも時間的な意味はありません。ここでは先頭48行に、2026年1月1日0時から1時間間隔の日時を練習用に付けます。この合成日時から、実際の経時変化や装置の劣化を判断することはできません。

`pd.date_range('2026-01-01', periods=48, freq='h')`は指定日時から48個の時刻を作ります。periodsは個数、freqは間隔で、hは1時間、30minは30分です。作った列を`set_index('measured_at')`で行ラベルにすると、DatetimeIndexという日時型の行ラベルになります。実データの日時が文字列なら、`pd.to_datetime(..., errors='coerce')`で日時型へ変え、変換できなかったNaTを調べてから使用します。

#### 時間枠の中をまとめるresample

`resample('6h')`は行を6時間ごとの枠へ分けます。続けてmeanやaggを使うと、各枠の平均や件数を計算できます。例題では`closed='left', label='left'`を明示します。0時の行ラベルは「0時以上6時未満」を表し、6時の測定は次の枠に入ります。境界の含め方と表示ラベルは別の指定なので、集計期間の意味を説明するときに確認します。

各枠の行数と有効件数を併記すると、6件あるはずの枠で平均が何件から計算されたかが分かります。平均は欠損を除くため、件数が異なる枠の値を同じ確かさで扱わないようにします。また、ここでは複数の生薬・成分を含む行へ機械的に日時を付けているので、平均はAPIの動作確認に限ります。

#### 時間軸だけを細かくするasfreq

`asfreq('30min')`は30分刻みの行ラベルへ置き換えます。平均を求める機能ではなく、元にない30分の行には欠損が入ります。0時から47時間後までなら、元の48行の間に47行が追加され、計95行になります。測定数が95件へ増えたわけではありません。

欠損を埋めるffillや補間を使うかは、その値が間の時刻でも保たれると考えてよいか、何のために集計するかによって決めます。HPLCの別々の検体の値を、時間的につながる同じ量だと決めつけて補間してはいけません。今回の練習では欠損を残し、元の欠損と間隔の変更によってできた欠損を区別します。

#### 例題（演習とは条件が異なります）

In [ ]:
clock = df.head(48).copy()
clock['measured_at'] = pd.date_range('2026-01-01', periods=48, freq='h')
clock = clock.set_index('measured_at').sort_index()
display(clock[['sample_id', 'retention_min']].head(3))
six_hour = clock.resample('6h', closed='left', label='left').agg(rows=('sample_id', 'size'), valid=('retention_min', 'count'), mean_retention=('retention_min', 'mean'))
display(six_hour)
half_hour = clock[['retention_min']].asfreq('30min')
print('元の行数 / 30分刻みの行数:', len(clock), len(half_hour))
display(half_hour.head(5))
print('日時は練習用。CSVの実際の測定時刻や経時変化を示さない。')

6時間集計は8行です。0時・6時・12時・18時というラベルは各区間の左端で、区間の右端は次の枠に入ります。保持時間には欠損がないため各枠のrowsとvalidは6です。asfreqでは30分、1時間30分などの新しい時刻に欠損が現れます。

#### コードを変えて試す

- resampleを12hに変え、集計行数と1枠の件数を確かめてください。
- asfreqの間隔を15minに変え、行数と新しくできる欠損の位置を確かめてください。

#### よくある間違い

- 合成した時刻から装置の時間変化を結論づける
- asfreqが自動的に平均や補間をすると思う
- 日付の表示形式が日時型への変換と同じだと思う

### 演習 PD-17

dfの先頭48行のコピーに、2026年1月1日0時から1時間間隔で48個の合成日時をmeasured_at列として付け、日時を行ラベルにしてください。6時間ごとに左端を含み右端を含まない枠へ分け、ラベルは左端とします。各枠のrows（sample_idの行数）、valid（content_mg_gの非欠損件数）、mean_content（content_mg_gの平均）をこの順に持つ8行の表をpd_17に入れてください。欠損は補いません。TODOセルの下のコメントには、validが6未満の枠で平均が何件から計算されるかと、合成日時の分析上の限界を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

pd.date_rangeで48個の日時を作り、表へ追加してからset_indexします。

</details>

<details><summary>ヒント2</summary>

resampleの間隔は6h、closedとlabelはどちらもleftです。

</details>

<details><summary>ヒント3</summary>

名前付きaggでsize・count・meanを使います。content_mg_gの欠損はcountとmeanから除かれます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_17 に入れてください。
pd_17 = None

In [ ]:
from learning import check_answer
check_answer('PD-17', globals().get('pd_17'))

#### 自己確認と振り返り

- 0時の集計には6時ちょうどの測定が入るか
- asfreqで増えた行は新たな観測といえるか
- このデータでは時間について何が言えないか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：時系列](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/timeseries.html)
- [pandas 2.3：resample](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.resample.html)
- [pandas 2.3：asfreq](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.asfreq.html)

### PD-18　移動平均・前回との差・変化率を計算する

目安：40分 / 前提：PD-17の時系列、PD-03の欠損値

使う場面：順序を持つ測定値について、直前との差と過去の平均を分けて記述する方法を練習する。

扱う機能：`rolling`, `min_periods`, `shift`, `diff`, `pct_change`, `expanding`, `ewm`, `cumsum`, `interpolate`

#### 学習目標

- rollingの窓幅とmin_periodsを指定する
- shiftで過去の値だけから計算する
- 差・変化率・累積平均の意味を区別する

#### 直近の数件を、順番にまとめる

`rolling`は、対象とする範囲を1行ずつ移動しながら計算する機能です。この範囲を「窓」と呼びます。`rolling(3, min_periods=3).mean()`は、現在の行を含む直近3行の平均です。min_periodsは計算に必要な非欠損値の最小数で、3なら3点そろわない箇所は欠損になります。最初の2行だけでなく、窓の中に元の欠損がある箇所でも平均は欠損です。

例題ではカンゾウの保持時間の先頭12行に、練習用の1時間間隔の日時を付けます。欠損を含む窓を試すため、コピーした値の6行目だけを欠損に変更します。元のCSVとdfは変えません。演習は成分量を対象にし、元からある欠損を残します。CSVの並びは実際の測定順ではなく、別々の検体です。移動平均を計算できても、装置のドリフトや検体の変化を示すことにはなりません。実データで使うときは対象・順序・間隔を確かめ、時間順に並べます。rolling(3)の3は3時間でなく3行です。不規則な日時で時間幅を指定したい場合は、日時インデックスに対してrolling('3h')を使います。

#### 現在の値を使ってよい時点かを考える

`shift(1)`は値を1行後ろへずらします。現在の行には前の行の値が置かれ、先頭は欠損になります。引数freqを指定しないここでのshiftは、日時そのものを変更しません。`s.shift(1).rolling(3, min_periods=3).mean()`なら、現在の行の直前3行だけから平均を作れます。

現在の測定前に予測する用途では、現在の測定値を使った平均を特徴量に入れると、予測時に得られない情報を使うことになります。現在値を含む平均がすべて不適切という意味ではなく、記述目的の平滑化か、未来を予測する入力かで使える情報が変わります。center=Trueで窓を中央へ置く場合も、後の行が計算に含まれ得る点を確認します。

#### 差、割合、平均は別の情報

`diff()`は現在値から前の値を引くため、単位は元と同じで、例題の保持時間なら分、演習の成分量ならmg/gです。`pct_change(fill_method=None)`は「現在値÷前の値−1」という割合の変化を返し、0.1は10%を表します。名前にpctがあっても、10という百分率を返すわけではありません。fill_method=Noneは欠損を自動で補わない指定です。前の値が0なら比率は定義できないため、infなどの結果を通常の値として報告しないよう点検します。

`expanding(min_periods=1).mean()`は先頭からその行までの非欠損値を使う累積平均です。`ewm(span=3, adjust=False).mean()`は直近を重くする指数加重平均で、欠損のない連続部分では、この指定で新しい値の重みは0.5です。欠損をまたぐ場合の重みは欠損の扱いによって変わります。rollingと違って、3行より前の影響も残ります。これらも観測を増やす処理ではありません。窓幅や補完方針を変えたときは、どの行の値が計算に入るかを確認します。

累積の合計は`cumsum()`です。件数など足し合わせる意味がある量に使います。別々の生薬の成分量mg/gを単純に足しても、試料全体の成分総量にはなりません。`interpolate()`は前後の値から途中を補間します。次の例は一定間隔で滑らかに変わると仮定した練習用の温度列だけで動作を確認します。補間値は測定値ではなく、未来の値を参照する補間を予測の入力に使うとデータ漏洩になり得ます。

#### 例題（演習とは条件が異なります）

In [ ]:
s = df.loc[df['herb'] == 'カンゾウ', 'retention_min'].head(12).copy()
# 欠損を含む窓を試すため、コピーの6行目だけを練習用に欠損へ変えます。
s.iloc[5] = np.nan
s.index = pd.date_range('2026-01-01', periods=12, freq='h', name='practice_time')
s = s.sort_index()
window = pd.DataFrame({'retention': s, 'rolling3': s.rolling(3, min_periods=3).mean(), 'past3': s.shift(1).rolling(3, min_periods=3).mean(), 'previous': s.shift(1), 'difference': s.diff(), 'change_fraction': s.pct_change(fill_method=None)})
display(window.head(8))
display(pd.DataFrame({'retention': s, 'expanding': s.expanding(min_periods=1).mean(), 'ewm': s.ewm(span=3, adjust=False).mean()}).head(8))
print('日時と行順は練習用。別々の検体の値を、装置の時間変化として解釈しない。')
# 別に作った練習用の件数と温度。元の測定CSVの欠損は補いません。
counts = pd.Series([3, 2, 4], index=['午前', '午後', '夜間'])
display(counts.cumsum().rename('累積件数'))
temperature_demo = pd.Series([30.0, None, 34.0])
display(pd.DataFrame({'練習値': temperature_demo, '直線補間': temperature_demo.interpolate()}))

例題のretentionは保持時間（分）です。rolling3は3行目から、past3は4行目から計算できます。練習として6行目を欠損にしたため、その値を含む3行窓も欠損になります。differenceの単位は分、change_fractionは単位を持たない割合です。expandingとewmは計算に使う履歴と重みが異なります。演習では成分量を使うため、差の単位はmg/gになります。 別の練習例の累積件数は3・5・9、直線補間した中央の温度は32です。

#### コードを変えて試す

- rollingのmin_periodsを1に変更し、欠損周辺の結果が変わる理由を説明してください。
- rolling3とpast3の値を1行ずつ比べ、現在値を含むかどうかを確かめてください。

#### よくある間違い

- min_periodsを窓幅そのものと混同する
- shift(1)を付けずに「現在より前の3件」と説明する
- pct_changeの0.1を0.1%と読む
- 値が欠けた行を自動補完してから、補完なしの変化率と説明する

### 演習 PD-18

カンゾウのcontent_mg_gの先頭12行を取り出してコピーし、2026年1月1日0時から1時間間隔の合成日時をpractice_timeという名前の行ラベルにしてください。content（元の値）、past3（直前3行の平均、非欠損値が3個必要）、difference（前の行との差）、change_fraction（前の行からの割合の変化、欠損の自動補完なし）の順に4列を持つ12行の表をpd_18に入れます。元の欠損は残してください。TODOセルの下のコメントで、past3の先頭3行が欠損になる理由と、change_fractionが0.1なら何%かを説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

Seriesをcopyし、そのindexにdate_rangeの結果を代入できます。

</details>

<details><summary>ヒント2</summary>

past3はshift(1)を先に行い、その後でrolling(3, min_periods=3).mean()を使います。

</details>

<details><summary>ヒント3</summary>

DataFrameへ列名とSeriesの辞書を渡して4列をまとめます。差はdiff、割合はpct_change(fill_method=None)です。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_18 に入れてください。
pd_18 = None

In [ ]:
from learning import check_answer
check_answer('PD-18', globals().get('pd_18'))

#### 自己確認と振り返り

- past3の4行目は元の何行目から何行目を使うか
- 欠損の直後のdiffや変化率が欠損になる理由は何か
- 記述する時点と予測する時点で、使える情報はどう変わるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：移動計算](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/window.html)
- [pandas 2.3：割合の変化](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.pct_change.html)
- [pandas 2.3：shift](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.shift.html)

### PD-19　分布・相関・順位を調べ、代表値を読み取る

目安：40分 / 前提：PD-02のdescribe、PD-13の集計

使う場面：同じ生薬の成分量の分布を記述し、極端な測定と条件間の関係を点検する。

扱う機能：`median`, `quantile`, `std`, `corr`, `rank`, `sample`, `nlargest`, `value_counts`

#### 学習目標

- 中央値・四分位点・標準偏差で分布を記述する
- 相関の対象行と解釈の限界を確認する
- 順位・上位抽出・無作為抽出を目的で使い分ける

#### 平均だけでは分からない分布を調べる

同じ平均でも、値が近くに集まる場合と大きく散らばる場合があります。例題ではショウキョウだけに絞り、6-gingerolの成分量を調べます。演習では対象をカンゾウへ変え、グリチルリチンの成分量を調べます。生薬が違えば測定成分も違うため、ここでは異なる成分をひとつの分布としてまとめません。`median()`は小さい順に並べた中央の値、`quantile([0.25, 0.5, 0.75])`は25%・50%・75%の分位点です。分位点は並べたデータのどの位置までを含むかの目安で、pandasの既定では必要に応じて隣接する値の間を補間します。

`std(ddof=1)`は、平均からの差の二乗和を非欠損件数nより1少ない数で割り、平方根を取る標準偏差です。単位は元と同じmg/gになります。pandasの既定はddof=1ですが、NumPyのstdの既定はddof=0です。同じ名前の関数でも指定が違えば値が変わる点を覚えておきましょう。標準偏差をそのまま規格範囲や平均の誤差と呼ぶことはできません。

#### 相関を読むときは、組にした列と対象行を確認する

`corr()`は数値列同士の相関係数を返します。既定のPearson相関は直線的な関係の強さを−1から1で表し、負なら一方が大きいと他方が小さくなる傾向です。0に近くても、曲線的な関係がないとは限りません。例題では流速・保持時間など、比較する列を明示します。

欠損がある場合は、列の組ごとに両方がそろった行だけを使います。そのため、相関表のすべてのセルが同じ件数から求められるとは限りません。`dropna(subset=[比較する2列])`で対象行を確認できます。ショウキョウ内で流速と保持時間に負の相関があっても、温度や溶媒比率などの条件はそろえていません。この表だけで流速の因果的な効果を測ったとは言えません。

#### 大きい値と、無作為に選んだ値を分ける

`nlargest(5, 'content_mg_g')`は成分量が大きい5行を取り出します。極端な値の点検に向いていますが、全体を代表する5行ではありません。`rank(ascending=False, method='min')`は大きい順に順位を付け、同じ値には同順位の最小番号を付けます。例えば12・12・10なら1位・1位・3位です。既定のmethod='average'では同率に平均順位を付けます。

`sample(n=5, random_state=42)`は5行を無作為に選びます。random_stateを固定すると同じ条件で再現できますが、小さな標本が全体を正確に代表する保証はありません。装置の構成比を確認する`value_counts(normalize=True)`は、既定では欠損を除いた件数を分母にします。欠損を含む構成を知りたければdropna=Falseを指定し、何を分母にした比率かを併せて伝えます。

#### 例題（演習とは条件が異なります）

In [ ]:
shokyo = df.loc[df['herb'] == 'ショウキョウ'].copy()
content = shokyo['content_mg_g']
print('有効件数:', content.count(), '中央値:', content.median(), '標準偏差:', content.std(ddof=1))
display(content.quantile([0.25, 0.5, 0.75]))
correlation = shokyo[['flow', 'retention_min', 'content_mg_g']].corr()
display(correlation)
print('流速と保持時間がそろう件数:', len(shokyo.dropna(subset=['flow', 'retention_min'])))
display(shokyo.nlargest(5, 'content_mg_g')[['sample_id', 'content_mg_g']])
display(pd.Series([12.0, 12.0, 10.0]).rank(ascending=False, method='min'))
display(shokyo.sample(n=5, random_state=42)[['sample_id', 'content_mg_g']])
display(shokyo['instrument'].value_counts(normalize=True, dropna=False))

例題のショウキョウ120行では、成分量が記録されている114行で分位点と標準偏差を計算します。流速と保持時間の相関は約−0.843で、この対象内では負の直線的な関係があります。因果的な効果の大きさではありません。上位5行と無作為5行は選び方が異なります。演習ではカンゾウを対象にするので、件数や相関係数を改めて求めます。

#### コードを変えて試す

- stdのddofを0へ変え、値がわずかに変わる理由を式の分母から説明してください。
- rankのmethodをaverageへ変え、同じ値に付く順位を比較してください。

#### よくある間違い

- 標準偏差と標準誤差、または規格範囲を同じものとして扱う
- 相関表の各セルは必ず同じ件数から計算されると思う
- 成分量の上位5件を全体の代表サンプルと呼ぶ
- normalize=Trueの分母に欠損が含まれるかを確認しない

### 演習 PD-19

カンゾウの全行を対象に、成分量のcount・median・q25・q75・stdという名前の順で5要素を持つSeriesを作ってください。q25とq75は25%点と75%点、stdはddof=1の標準偏差とします。さらに成分量の大きい順の上位5行からsample_idとcontent_mg_gの2列を取り出し、流速flowと保持時間retention_minのPearson相関係数も求めてください。pd_19をsummary（Series）・top5（表）・correlation（数値）の辞書にします。欠損値は補わず、各計算で利用可能な値を使います。TODOセルの下のコメントには、相関係数の符号から読めることと、因果関係として言えないことを書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

生薬で抽出した後、成分量列へcount・median・quantile・stdを使います。

</details>

<details><summary>ヒント2</summary>

pd.Series({名前: 値})で5項目の要約をまとめ、nlargest(5, 列名)で上位行を選びます。

</details>

<details><summary>ヒント3</summary>

2列のcorr()が返す表から.loc['flow', 'retention_min']で1つの相関係数を取り出せます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を pd_19 に入れてください。
pd_19 = None

In [ ]:
from learning import check_answer
check_answer('PD-19', globals().get('pd_19'))

#### 自己確認と振り返り

- 中央値と平均が違うとき、次にどんな図や値を調べるか
- 負の相関があっても断定できないことは何か
- 同順位の扱いと抽出方法を報告時に説明できるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3：記述統計などの基本操作](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/basics.html)
- [pandas 2.3：標準偏差](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.std.html)
- [pandas 2.3：相関](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.corr.html)
- [pandas 2.3：順位](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.rank.html)

### PD-20　関数とmap・apply・pipeで処理をまとめる

目安：35分 / 前提：PD-01〜PD-19の説明。前の演習の回答は不要です。

使う場面：月ごとのCSVに同じ単位変換と装置名の置き換えを適用し、処理の違いを減らしたいときに使います。

扱う機能：`def`, `Series.map`, `DataFrame.apply`, `assign`, `pipe`

#### 学習目標

- 1列・1行・表全体のどれを処理するかで関数の使い方を選ぶ
- 元の表を変えない加工関数を作る
- 列全体の計算とapplyの役割を区別する

#### 同じ処理に名前を付ける

保持時間を秒に直す式を毎回書くなら、関数にしておくと変更箇所を一つにできます。`def add_seconds(frame):`は関数の定義です。`frame`には呼び出し時の表が入り、字下げした処理を行います。`return`は加工した表を返します。`assign`は指定した列を加えた新しい表を返すので、入力を直接書き換えずに使えます。

#### map・apply・pipeは何を渡すかが違う

| 操作 | 関数や対応表に渡るもの | 今回の用途 |
|---|---|---|
| `Series.map` | 1列の各値 | 装置コードから表示名へ |
| `DataFrame.apply(..., axis=1)` | 1行を表すSeries | 複数列を参照する独自の計算 |
| `DataFrame.apply(..., axis=0)` | 1列を表すSeries | 列ごとの独自集計 |
| `DataFrame.pipe` | 表全体 | 名前の付いた加工工程を順に適用 |

辞書の`map`で対応表にない値は欠損になります。必ず置換後に未対応がないか確かめます。ここでは元データにLC-AとLC-Bだけがあると分かっているため、2つの対応を定義します。

#### 列全体で計算できる式は、そのまま書く

例題の送液量は`保持時間 × 流速`です。流速が一定と仮定した計算練習で、実験の消費溶媒量を測った値ではありません。行ごとの`apply`でも書けますが、列同士の掛け算の方が式が短く、一般に大量データでも処理しやすくなります。`lambda row: ...`は、小さな関数をその場で定義する書き方です。`row`が受け取る1行、コロンの右が返す値です。

`pipe`は自動的に速くする機能ではありません。処理の順番に名前を付け、各工程を個別に確認できることが利点です。大きな式を一度に作らず、最初は途中結果の列名と先頭行を表示してください。

#### 例題（演習とは条件が異なります）

In [ ]:
def add_seconds(frame):
    return frame.assign(retention_sec=frame['retention_min'] * 60)

labels = {'LC-A': '装置A', 'LC-B': '装置B'}
example = df.head(4).pipe(add_seconds)
example = example.assign(instrument_label=example['instrument'].map(labels))
example['volume_ml'] = example['retention_min'] * example['flow']
row_result = example.apply(lambda row: row['retention_min'] * row['flow'], axis=1)
display(example[['sample_id', 'instrument_label', 'retention_sec', 'volume_ml']])
print('列計算と行計算は一致:', np.allclose(example['volume_ml'], row_result))
print('元の表には新列がない:', 'retention_sec' not in df.columns)

送液量の2通りの計算は同じになります。元のdfにretention_secがないことも確認できます。行計算の結果だけを見るのでなく、入力の列を変えていないかも確認しましょう。

#### コードを変えて試す

- labelsからLC-Bの対応を外し、欠損になる行を確認してから元に戻す。
- add_secondsの変換先をミリ秒にした別関数を作り、単位を列名にも書く。

#### よくある間違い

- axis=1は行ごと、axis=0は列ごとです。NumPyの集計の感覚だけで選ばず、関数が受け取るものを確認します。
- mapで生じた欠損を元からあった欠損と混同しないでください。
- 単純な列計算にもapplyを使う必要はありません。

### 演習 PD-20

全360行について、列がsample_id、instrument_label、retention_secの順のDataFrameをpd_20に入れてください。装置名はLC-A→分析装置A、LC-B→分析装置B、保持時間は分から秒へ変換します。秒への変換を行う関数を定義し、pipeで適用してください。元のdfを変更せず、先頭5行と変換後の欠損件数を表示します。装置名変換をmapで行うとき、未対応のコードが来たらどうなるか説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

まずdefで入力の表と返す表を決めます。戻り値にはassignを使えます。

</details>

<details><summary>ヒント2</summary>

装置名は2項目の辞書を作ってSeries.mapへ渡します。

</details>

<details><summary>ヒント3</summary>

加工の後で必要な3列を順番に選択します。isna().sum()で未対応を確認します。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_20 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-20', globals().get('pd_20'))

#### 自己確認と振り返り

- 自分の関数が入力を書き換えるか、新しい表を返すか説明できますか。
- 同じ式を列計算で書けるなら、applyを選ぶ理由はありますか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/basics.html)

### PD-21　CSVの読込条件とメモリ使用量を調整する

目安：35分 / 前提：PD-01〜PD-20の説明。前の演習の回答は不要です。

使う場面：装置から出したCSVの列が多い、区切りが異なる、ファイルが大きいときに、読み込み方を明示します。

扱う機能：`read_csv`, `usecols`, `dtype`, `na_values`, `encoding`, `chunksize`, `memory_usage`

#### 学習目標

- 区切り・欠損表記・IDの型を指定して読む
- 必要な列だけを選び、メモリ量を確認する
- 小分けに読んでも正しい全体平均を計算する

#### 装置の出力形式を先に確かめる

CSVは型や単位を厳密に保存する形式ではありません。`read_csv`は値を見て型を推測します。番号の先頭の0を残すなら`dtype={'sample_id': 'string'}`のように指定します。`sep`は区切り文字、`encoding`は文字とバイトの対応規則です。同梱CSVはUTF-8なので`encoding='utf-8'`で読めます。文字化けした場合は出力元の設定を確かめ、根拠なく文字を捨てて読み込まないでください。

`na_values=['未記録']`は追加の欠損表記を指定します。「ND」が不検出を示す業務なら、欠損と同じ意味ではありません。どの表記を空欄として扱うかは、元の記録の定義に合わせます。例題ではPythonの`StringIO`を使い、文字列を小さなファイルのように読みます。これは装置ファイルの読込設定を試すための作り物です。

#### 必要な列だけを読む

`usecols`は読込対象列を絞ります。ただし指定したリストの順を必ず保存する機能ではないため、列順が必要なら読込後に選び直します。`memory_usage(deep=True).sum()`は表が使うメモリの概算で、Python全体の使用量やファイルサイズとは違います。頻出する少数の文字列はcategory型で小さくできる場合があります。

#### 小分けに読むときの平均

`chunksize=100`を指定すると、表が100行ずつ返ります。`for chunk in reader:`は表を一つずつ受け取る繰り返しです。`total += ...`は合計を足して更新します。最後の60行も忘れず処理してください。

平均は全ブロックの合計値を全有効件数で割ります。ブロックごとの平均を単純に平均すると、ブロックの長さや欠損件数の差を無視します。全値が欠損なら割る件数が0になるため、条件分岐も必要です。データは360行なので速度比較はせず、まず一括読込と同じ結果になるか確かめます。

#### 例題（演習とは条件が異なります）

In [ ]:
from io import StringIO
text = 'sample_id;flow\n0001;0.70\n0002;未記録\n'
preview = pd.read_csv(StringIO(text), sep=';', dtype={'sample_id': 'string'}, na_values=['未記録'])
display(preview)
small = pd.read_csv('data/hplc.csv', usecols=['sample_id', 'herb', 'flow'], dtype={'sample_id': 'string'})
print('選んだ3列のメモリ概算 (bytes):', small.memory_usage(deep=True).sum())
total, valid = 0.0, 0
for chunk in pd.read_csv('data/hplc.csv', usecols=['flow'], chunksize=100):
    total += chunk['flow'].sum()
    valid += chunk['flow'].count()
print('流速の有効件数:', valid)
print('全体の平均流速:', total / valid if valid else None)

0001の先頭の0が残り、未記録が欠損になっています。流速は欠損がないので有効件数は360です。メモリ量は環境で多少変わるため、固定値として覚えず、列や型を変えたときの差を見ます。

#### コードを変えて試す

- chunksizeを80、128に変え、平均が変わらないことを確かめる。
- herbをcategoryに変え、memory_usageの差を確認する。

#### よくある間違い

- CSVの行数と有効件数は、欠損がある列では一致しません。
- 小分けにした表をすべてリストにためると、省メモリの目的を失います。
- sepやencodingが違うとき、後の整形で直す前に読込設定を確認します。

### 演習 PD-21

content_mg_gだけをchunksize=80で読み込み、全体の有効件数valid、欠損件数missing、有効値の平均meanを辞書pd_21に入れてください。最後の端数ブロックも処理し、一括読込の平均と照合します。各ブロックの平均を単純に平均してはいけない理由と、異なる成分をまとめた全体平均の解釈上の限界をコメントで説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

合計・有効件数・欠損件数の変数を0で初期化します。

</details>

<details><summary>ヒント2</summary>

各ブロックのsum、count、isna().sum()を加えます。

</details>

<details><summary>ヒント3</summary>

平均は合計÷有効件数です。有効件数0ならNoneなどの扱いを決めます。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_21 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-21', globals().get('pd_21'))

#### 自己確認と振り返り

- 欠損を0として平均すると、どこが変わりますか。
- 中央値や厳密な重複除去も、同じ3つの累積値だけで求められますか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/scale.html)

### PD-22　ExcelとJSONを読み書きする

目安：35分 / 前提：PD-01〜PD-21の説明。前の演習の回答は不要です。

使う場面：測定表と成分マスタを一つのExcelファイルにまとめたり、別のプログラムにJSONで渡したりするときに使います。

扱う機能：`ExcelWriter`, `read_excel`, `to_excel`, `read_json`, `to_json`, `json_normalize`

#### 学習目標

- 複数シートのExcelを作成して指定シートを読み直す
- JSONの行形式を選び、保存前後の値を確かめる
- 入れ子のJSONを表にする

#### 形式を変えるときも1行の意味を保つ

Excelは複数の表をシートに分けられます。JSONは名前と値の組やリストを入れ子にできるテキスト形式です。ファイル形式を変えても、列の単位や欠損の意味は別途伝える必要があります。出力後に読み直して、行数・列名・値を照合しましょう。

#### ExcelWriterで書込先をまとめる

`with pd.ExcelWriter(..., engine='openpyxl') as writer:`は、ブロック内で同じ書込先を使い、終わると保存を完了する書き方です。`with`で管理する対象は処理の後に閉じられます。表ごとに`sheet_name`を指定し、不要な行番号を書かないよう`index=False`を付けます。

`read_excel(..., sheet_name='measurements')`は指定シートを1表として読みます。`sheet_name=None`なら、シート名をキー、表を値とする辞書です。`header`や`usecols`で表の開始行や必要列を指定できます。セルの色や結合セル、グラフの編集をDataFrameがそのまま引き継ぐわけではありません。教材は自分で作った小さな表だけを扱います。openpyxlは初回セットアップで導入済みです。

#### JSONはorientを合わせる

`orient='records'`は、1行を1つの辞書として並べる形式です。列名は残りますが、インデックスやpandasの型をすべて保存する形式ではありません。`force_ascii=False`を指定すると、日本語を読みやすい文字のまま書けます。日付を含む場合は`date_format='iso'`などを検討し、読込後の型を確かめます。

例題の`json_normalize`は、手作りの入れ子の辞書を1段の表にします。`result.area`のような列名は、元の階層をドットで示したものです。外部サービスには接続しません。保存後に整数と小数の型が変わることがあるので、用途に必要な一致条件を決めることが大切です。

#### 例題（演習とは条件が異なります）

In [ ]:
example = df[['sample_id', 'herb', 'flow']].head(6)
with pd.ExcelWriter('outputs/example_measurements.xlsx', engine='openpyxl') as writer:
    example.to_excel(writer, sheet_name='measurements', index=False)
    analytes.to_excel(writer, sheet_name='analytes', index=False)
book = pd.read_excel('outputs/example_measurements.xlsx', sheet_name=None)
print('シート名:', list(book))
display(book['measurements'].head(3))
example.to_json('outputs/example_measurements.json', orient='records', force_ascii=False, indent=2)
display(pd.read_json('outputs/example_measurements.json', orient='records').head(3))
nested = [{'sample_id': 'DEMO-1', 'result': {'area': 12345, 'unit': '任意単位'}}]
display(pd.json_normalize(nested))

Excelにはmeasurementsとanalytesの2シートがあります。JSONの読み戻しでも6行3列になります。DEMO-1は入れ子構造を説明する別の練習データです。表示が似ていても、インデックスや型まで同じとは限りません。

#### コードを変えて試す

- sheet_nameをanalytesに変え、マスタの3行だけを読む。
- orient=recordsとorient=tableで保存したJSONをテキストとして開き、型に関する情報の違いを探す。

#### よくある間違い

- 既存の業務ファイル名を出力先にすると上書きします。教材ではoutputs配下だけを使います。
- ExcelWriterのwithブロック内で読み直さず、保存が終わってから読みます。
- JSONのorientを保存時と読込時で無関係に変えないでください。

### 演習 PD-22

装置別にtemperatureの有効件数validと平均mean_temperatureを集計し、列をinstrument、valid、mean_temperatureの順にしてください。outputs/pd22_summary.xlsxのsummaryシートとoutputs/pd22_summary.json（orient=records）へ保存します。Excelから読み直した表をpd_22に入れ、JSONも読み直して値を比較してください。有効件数が装置別の全行数と異なる理由を説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

groupbyの名前付きaggでvalid=(temperature,count)、mean_temperature=(temperature,mean)を指定します。

</details>

<details><summary>ヒント2</summary>

reset_indexで装置名を列に戻し、Excelはindex=Falseで保存します。

</details>

<details><summary>ヒント3</summary>

小数の保存誤差があるので、比較にはnp.allcloseなどを使えます。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_22 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-22', globals().get('pd_22'))

#### 自己確認と振り返り

- measurementsとanalytesを分けると、行の単位をどう保てますか。
- 見た目の表示桁数と、保存された数値の精度は同じですか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/io.html)

### PD-23　SQLiteとParquetで表を保存する

目安：40分 / 前提：PD-01〜PD-22の説明。前の演習の回答は不要です。

使う場面：ローカルで表に条件を付けて取り出したり、型を保ちながら分析用データを保存したりするときに使います。

扱う機能：`sqlite3.connect`, `to_sql`, `read_sql_query`, `to_parquet`, `read_parquet`

#### 学習目標

- 外部サーバーを使わずSQLiteとpandasの間で表を渡す
- SQLの選択・抽出とパラメータの役割を理解する
- Parquetから必要な列を読み戻す

#### SQLiteとParquetの用途を分ける

SQLiteは、表を保存してSQLという問い合わせ文で取り出す仕組みです。Python標準の`sqlite3`が使えます。Parquetは列を中心に保存するファイル形式で、分析用の表を型とともに保存する用途に向きます。読み書きには初回導入済みのpyarrowを使います。CSVは目で開いて点検しやすいので、用途に応じて選びます。360行だけで形式の速度優劣は判断しません。

#### SQLの最小限の読み方

`SELECT sample_id, flow`は列を選び、`FROM measurements`は表を指定します。`WHERE flow >= ?`は条件抽出、`ORDER BY sample_id`は並べ替えです。SQLの`?`には`params`で値を渡します。文字列をつなげて問い合わせを作らないため、値と命令を区別できます。

例題の`sqlite3.connect(':memory:')`は一時的なデータベースをメモリ内に作り、接続を閉じると消えます。`closing`は接続を最後に閉じるためのPythonの補助機能です。永続化したい場合は、自分の`outputs/pd23_example.sqlite`のようなファイルを指定できます。`to_sql`の`if_exists='replace'`は同名の表を置き換えるので、業務の保存先で試さないでください。教材では一時的な表だけを置き換えます。

#### 保存前と読み戻し後を比較する

`to_parquet(..., index=False)`で行番号を除いて保存し、`read_parquet(..., columns=[...])`で必要な列を読みます。欠損を持つ数値列も扱えますが、別ソフトへ渡すときは型や日時の表現を読み戻して確認します。配列や複雑なPythonオブジェクトを何でもそのまま保存できる形式ではありません。

SQLには`GROUP BY`による集計や結合もありますが、このレッスンでは表の受け渡しに集中します。集計の正しさは、既に学んだpandasで同じ条件の表を作り、行数・ID・数値を照合して確認できます。

#### 例題（演習とは条件が異なります）

In [ ]:
import sqlite3
from contextlib import closing
with closing(sqlite3.connect(':memory:')) as connection:
    df.to_sql('measurements', connection, index=False, if_exists='replace')
    query = 'SELECT sample_id, flow FROM measurements WHERE flow >= ? ORDER BY sample_id'
    selected = pd.read_sql_query(query, connection, params=(0.9,))
display(selected.head())
print('SQLで選んだ件数:', len(selected))
df.to_parquet('outputs/example_hplc.parquet', engine='pyarrow', index=False)
back = pd.read_parquet('outputs/example_hplc.parquet', columns=['sample_id', 'temperature'])
display(back.head())
print('温度の欠損件数:', back['temperature'].isna().sum())

SQLでも同じ流速の条件で行を選べます。params=(0.9,)の末尾のカンマは、1要素のタプル（値の組）を作るためです。Parquetの読み戻しには360行あり、温度の欠損18件も保存されています。

#### コードを変えて試す

- paramsの流速を0.95に変え、pandasの条件抽出と件数を比べる。
- Parquetをcolumnsなしで読み、全列の型と元データの型を並べて見る。

#### よくある間違い

- SQLはORDER BYを指定しないと行順を保証しません。
- if_exists=replaceを使うと既存の同名表は置き換わります。
- Parquetをテキストエディタで読もうとせず、read_parquetで内容を確認します。

### 演習 PD-23

SQLiteの一時データベースへdfを書き込み、herbがカンゾウのsample_id、herb、retention_minをsample_id昇順で取得してください。値はparamsで指定します。結果をoutputs/pd23_kanzo.parquetへindex=Falseで保存し、読み戻した表をpd_23に入れてください。pandasの条件抽出と行数・ID・数値を比較し、SQLでORDER BYが必要な理由を説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

例題のWHEREをherb = ?にし、paramsにはカンゾウを1要素のタプルで渡します。

</details>

<details><summary>ヒント2</summary>

SQLのSELECTで3列を順番に指定します。

</details>

<details><summary>ヒント3</summary>

Parquetを読み直し、元のdfを条件抽出・整列した表と比べます。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_23 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-23', globals().get('pd_23'))

#### 自己確認と振り返り

- 値をparamsで渡す場合とSQL文へ直接つなぐ場合は、何が違いますか。
- CSV、Excel、SQLite、Parquetをどんな相手・用途で選びますか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/io.html)

### PD-24　pandasからグラフと読みやすい表を作る

目安：35分 / 前提：PD-01〜PD-23の説明。前の演習の回答は不要です。

使う場面：集計の直後に大きさの違いを確認し、単位と件数を添えた表を報告に使いたいときに役立ちます。

扱う機能：`DataFrame.plot`, `Series.plot.bar`, `Styler.format`, `Styler.set_caption`, `Styler.to_html`, `option_context`

#### 学習目標

- pandasのplotで棒グラフを作り、軸と単位を付ける
- Stylerで表示桁数を整え、元の数値を保持する
- 表示設定を一時的に変更する

#### まず、何を比較する図かを決める

pandasの`plot`はMatplotlibを使って図を描く入口です。`Series.plot.bar()`はインデックスを横軸、値を棒の高さにします。`kind='line'`、`kind='hist'`などで図の種類を変えられます。散布図は`DataFrame.plot.scatter(x='flow', y='retention_min')`のように2列を指定します。折れ線は並び順に意味があるとき、ヒストグラムは分布、棒はカテゴリの比較に使います。

ここでは生薬ごとの成分量の欠損件数を棒で示します。母数は各120件です。欠損を数えた表も併記するので、色だけに依存せず値を確認できます。棒グラフの縦軸は0から始めます。

#### 図全体と描画領域

`plot`が返す`ax`は、座標軸とデータを描く領域です。図全体は`ax.figure`で参照できます。`ax.set_xlabel`、`ax.set_ylabel`で軸の意味と単位を設定し、`ax.figure.tight_layout()`でラベルの余白を調整します。`plt.show()`は図を表示する命令です。詳しいFigureとAxesの使い分けは後のMatplotlib章で扱います。

`setup_plotting()`は同梱フォントを登録します。初回セットアップ以外のダウンロードは不要です。グラフの形だけで、欠損が発生した理由や装置の優劣は分かりません。

#### 表示だけを変える

`table.style.format`は表示する小数桁数を指定します。計算に使う表そのものを丸める`round`とは役割が違います。`set_caption`に対象と単位を記すと、保存した表だけでも読みやすくなります。`to_html`で保存するHTMLは表の出力で、データの正本はCSVなどに別途残します。

`pd.option_context`はブロック内だけ表示設定を変える書き方です。表を全部表示する設定を永続的に変えてしまうより、必要な場所に限定できます。表示桁数を増やしても、測定の精度が上がるわけではありません。

#### 例題（演習とは条件が異なります）

In [ ]:
import matplotlib.pyplot as plt
from learning import setup_plotting
setup_plotting()
summary = df.groupby('herb').agg(total=('sample_id', 'size'), valid=('content_mg_g', 'count'))
summary['missing'] = summary['total'] - summary['valid']
ax = summary['missing'].plot.bar(color='#006b5e', rot=0, title='架空測定：生薬別の成分量の欠損件数')
ax.set_xlabel('生薬（各120件）')
ax.set_ylabel('欠損件数（件）')
ax.set_ylim(bottom=0)
ax.figure.tight_layout()
plt.show()
plt.close(ax.figure)
display(summary.style.set_uuid('pd24example').format(precision=0).set_caption('架空測定360件の記録状況'))
with pd.option_context('display.precision', 2):
    display(df[['flow', 'retention_min']].head(3))

生薬ごとに欠損件数が違って見えても、原因をこの表から決めることはできません。各行のtotalは120、validとmissingの和も120です。表示設定を変えてもdf内の保持時間の数値は変わりません。

#### コードを変えて試す

- 棒をbarhにして横向きにし、軸名も入れ替える。
- 成分量のヒストグラムを一つの生薬だけで描く。異なる成分を一緒にする場合との意味の違いを考える。

#### よくある間違い

- 表示桁数の変更を測定精度の変更と混同しないでください。
- 集計済みの平均だけでは、各群のばらつきは分かりません。
- 図の色だけで判断させず、軸・単位と元の集計値を載せます。

### 演習 PD-24

装置別にflowの平均を集計してSeriesを作り、0始まりの棒グラフを描いてください。横軸を装置、縦軸を平均流速（mL/min）とし、図のタイトルに架空測定と明記します。辞書pd_24にsummaryとして集計Series、figureとして図全体を入れます。平均を小数3桁で表示するStyler表も作り、outputs/pd24_summary.htmlに保存します。この図だけでは装置の性能差を判断できない理由をコメントに書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

df.groupby(instrument)[flow].mean()でSeriesを作ります。

</details>

<details><summary>ヒント2</summary>

Series.plot.barが返すaxからax.figureを受け取れます。

</details>

<details><summary>ヒント3</summary>

Seriesをto_frameで表にしてstyle.formatを使います。保存はStyler.to_htmlです。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_24 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-24', globals().get('pd_24'))

#### 自己確認と振り返り

- 図を見ずに、タイトルと軸名だけで対象と単位が分かりますか。
- 差の原因を判断するには、どの条件や記録が追加で必要ですか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/visualization.html)

### PD-25　処理結果を検証し、差分を調べる

目安：35分 / 前提：PD-01〜PD-24の説明。前の演習の回答は不要です。

使う場面：加工コードを修正した後に、行が欠けていないか、値が意図せず変わっていないかを確認します。

扱う機能：`assert`, `equals`, `testing.assert_frame_equal`, `compare`, `is_unique`

#### 学習目標

- 行数・キー・欠損など守る条件をassertで記す
- 表全体の比較と変更箇所の表示を使い分ける
- 小さな既知のデータで計算を検証する

#### エラーが出ないことと、正しいことは別

結合で行が増えても、列を取り違えても、コードは動くことがあります。処理の途中に「検体IDは一意」「入力の行数を保つ」「欠損を除いた件数が合う」などの条件を置きます。`assert 条件, '説明'`は条件がFalseなら処理を止めて説明を示すPython構文です。ここでは学習用の確認に使います。安全に関わる入力検査をassertだけに任せる設計はしません。

#### 表が同じかを調べる

`equals`は表の値・形・ラベルと各列のデータ型が一致するかをTrue/Falseで返します。同じ位置の欠損は同じと扱います。行・列ラベルの型は、値が等しければ異なっていてもTrueになる場合があります。`pd.testing.assert_frame_equal`は、どの列・値が違うかもエラーで知らせます。小数には丸め誤差があるため、`check_exact=False, rtol=..., atol=...`で必要な許容差を決められます。`rtol`は値の大きさに比例する許容差、`atol`は固定の許容差です。

比較前に何でも並べ替えたり型を無視したりすると、本当の問題を見落とします。「順序まで同じ必要があるか」「型が変わってもよいか」を先に決めます。`compare`は同じ行・列ラベルを持つ表の差分を表示し、selfが元、otherが比較先です。

#### 小さな表で期待値を作る

例題では既知の1分と2.5分を秒へ変換し、60秒と150秒になるか確かめます。同じ計算式をコピーして期待値を作るだけでは、式自体の間違いを検出できません。単位から手で確かめられる小例と、本データの件数・キーの点検を組み合わせます。

CSVを保存したら、書込前後も比較します。エラーを意図的に起こす練習は別のセルで行い、配布Notebookの全実行が途中で止まらないよう、元に戻します。エラーの文を読んで、型の違いなのか値の違いなのかを区別しましょう。

#### 例題（演習とは条件が異なります）

In [ ]:
tiny = pd.DataFrame({'retention_min': [1.0, 2.5]})
actual = (tiny['retention_min'] * 60).to_frame('retention_sec')
expected = pd.DataFrame({'retention_sec': [60.0, 150.0]})
pd.testing.assert_frame_equal(actual, expected)
assert df['sample_id'].is_unique, '検体IDが重複しています'
assert len(df) == 360, '行数が変わっています'
print('小さな既知の例と、検体ID・行数の確認に通過しました。')
before = df[['sample_id', 'flow']].head(3).set_index('sample_id')
after = before.copy()
after.iloc[0, 0] = after.iloc[0, 0] + 0.01
display(before.compare(after))
print('変更前後が同じ:', before.equals(after))

差分には変更した最初の検体のflowだけが出ます。これは差分表示の練習用に値を変えたもので、元のCSVを書き換えていません。assert_frame_equalは成功したときに表を返さず、何も表示せずに次へ進みます。

#### コードを変えて試す

- tinyの期待値150.0を149.0にしてエラーを読み、確認後に150.0へ戻す。
- afterに元と同じ値を入れると、compareがどんな表になるか確認する。

#### よくある間違い

- assert_frame_equalの戻り値を回答表として使わないでください。成功時はNoneです。
- 小数の一致確認で許容差を大きくしすぎると、意味のある違いも見逃します。
- 表の行ラベルが異なるとcompareでは直接比べられません。

### 演習 PD-25

元のdfの先頭4行からsample_idとretention_minを取り出し、sample_idをインデックスにした表beforeを作ってください。そのコピーafterのS0002だけを1.0分増やします。before.compare(after)の結果をpd_25に入れてください。dfの行数360とsample_idの一意性をassertで確認し、差分のselfとotherの意味、元のdfを残す理由を説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

head(4)、2列の選択、set_indexを順に使います。

</details>

<details><summary>ヒント2</summary>

after=before.copy()で分け、loc[S0002,retention_min]にだけ加算します。

</details>

<details><summary>ヒント3</summary>

compareは変更箇所だけを返します。元の入力も変わったならコピーの作り方を見直します。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_25 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-25', globals().get('pd_25'))

#### 自己確認と振り返り

- 見た目が同じ表でもequalsがFalseになるとしたら、何を調べますか。
- 期待値を処理と同じ式から作るだけでは、どんな誤りを見落としますか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/merging.html)

### PD-26　章末演習：測定記録から集計報告を作る

目安：40分 / 前提：PD-01〜PD-25の説明。前の演習の回答は不要です。

使う場面：元の測定記録を受け取り、件数・欠損・測定条件を説明できる集計表として同僚へ渡す場面です。

扱う機能：`read_csv`, `isna`, `groupby.agg`, `merge`, `sort_values`, `to_csv`, `assert_frame_equal`

#### 学習目標

- 読込・点検・集計・保存を一連のコードにする
- 全件数と有効件数を分けて報告する
- 結果から分かることと分からないことを文章にする

#### 最初に、報告する単位を決める

生薬と装置の組合せごとに、記録件数と保持時間を整理します。元の表は1行が独立した検体の1成分、集計表は1行が生薬と装置の組合せです。行の単位が変わったら、sample_idをそのまま代表値のように載せず、件数を明示します。

成分量の欠損を除いてから全列を集計すると、保持時間は記録されているのに集計対象から消えてしまいます。今回の目的では行を一律に落とさず、列ごとに有効件数を数えます。集計に使わない温度の欠損も、元の記録の点検結果として別途確認できます。

#### 一度に全部書かず、途中を点検する

最初に列名と行数、検体IDの重複、欠損件数を見ます。次に生薬と装置でgroupbyし、全件数はsize、成分量の有効件数はcountで求めます。保持時間は平均だけでなく中央値も残し、差が大きい群があれば分布を追加で調べる候補とします。

例題では装置だけで集計して、保存後に読み直すところまで試します。演習は生薬と装置を組み合わせた6行の表に進みます。ほかの演習の変数は使わず、章の準備セルと今回のコードで実行できます。

#### 数字・条件・限界を一緒に書く

報告は「何を数えたか」「どの値をどうまとめたか」「この表では判断できないこと」の3点を含めます。たとえば欠損があったことは言えますが、その原因を測定失敗と断定はできません。装置ごとの平均保持時間が違っても、温度・流速・成分の構成が違うので装置の効果とは限りません。

各生薬の成分は異なります。成分量を全群で順位付けして薬効を比べることはしません。生成式を使った架空データであることも、報告の対象と限界の一部です。読み手が元の表を持っていなくても条件を理解できるか確認してください。

#### 例題（演習とは条件が異なります）

In [ ]:
raw = pd.read_csv('data/hplc.csv')
print('元の行数:', len(raw), 'IDの重複:', raw['sample_id'].duplicated().sum())
display(raw[['temperature', 'content_mg_g']].isna().sum())
summary = raw.groupby('instrument').agg(total=('sample_id', 'size'), valid_content=('content_mg_g', 'count'), mean_retention=('retention_min', 'mean')).reset_index()
summary['missing_content'] = summary['total'] - summary['valid_content']
summary.to_csv('outputs/example_report.csv', index=False)
back = pd.read_csv('outputs/example_report.csv')
pd.testing.assert_frame_equal(summary, back, check_exact=False, rtol=1e-10)
display(back)
print('全件数の合計:', summary['total'].sum())

装置別のtotalを足すと360です。valid_contentは合計336、missing_contentは24になります。保持時間の平均は全360行を使い、成分量の空欄を理由に保持時間の記録まで捨てていません。

#### コードを変えて試す

- groupbyのキーをherbだけに変え、欠損件数がどう見えるか比較する。
- 保持時間にminとmaxも加える。範囲が広いだけで異常値と断定できない理由を考える。

#### よくある間違い

- dropna()だけで全列の欠損行を落とすと、列ごとに異なる記録状況を失います。
- 集計表の6行と元の360行は1行の意味が異なります。
- CSVの保存だけで終えず、読み戻しと合計件数を確かめます。

### 演習 PD-26

data/hplc.csvを読み、生薬herbと装置instrumentごとに集計してください。列はherb、instrument、total（全件数）、valid_content（成分量の有効件数）、missing_content（成分量の欠損件数）、mean_retention（平均保持時間・分）、median_retention（中央値・分）の順にします。herb、instrumentの昇順に整えた6行の表をpd_26に入れ、outputs/pd26_report.csvに行番号なしで保存し、読み戻して照合してください。コメントで、件数・欠損について数値を使った説明、保持時間の読み取り、装置差や薬効を断定できない理由の3点を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

sizeは全件数、countは対象列の有効件数です。同じgroupbyで名前付きaggを使えます。

</details>

<details><summary>ヒント2</summary>

missing_contentはtotal-valid_contentです。列順を最後に明示します。

</details>

<details><summary>ヒント3</summary>

全件数360、有効件数336、欠損件数24になるか、群の合計でも確認してください。

</details>

In [ ]:
# TODO: 問題文の条件に合わせてコードを書きます。
pd_26 = None
# 解釈の問いへの回答も、この下にコメントで書いてください。

In [ ]:
from learning import check_answer
check_answer('PD-26', globals().get('pd_26'))

#### 自己確認と振り返り

- 欠損を除く単位を「行全体」から「対象列」に変えた理由を説明できますか。
- 再度同じCSVを受け取ったとき、手作業なしで同じ報告を作れますか。

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

#### 公式資料で調べる

- [pandas 2.3 公式ガイド](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/groupby.html)

## 2. NumPy

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)


### NP-01　表から数値配列へ移す

目安：25分 / 前提：pandasの列選択

#### 学習目標

- DataFrameと配列の違いを説明する
- shape・dtype・スライスを使う

pandasは列名や行ラベルを持つ表に向き、NumPyは同じ型の値が並ぶ配列を使った数値計算に向きます。列を選んで `to_numpy()` を呼ぶと、表から配列へ移せます。変換後は列名が付いていないため、どの位置がどの項目かを自分で管理します。

`shape`は各方向の要素数です。360行2列なら `(360, 2)` と表示されます。dtypeは値の型を表し、この例では小数を扱うfloat64です。温度と流速は単位も範囲も異なるため、並べたからといって同じ意味にはなりません。

配列の `a[0,1]` は0行目1列目、`a[:3,0]`は先頭3行の0列目です。終点を含まないスライスはpandasのilocと同じです。`a[a[:,1] > 0.8]` は2列目で作った条件を行の選択に使います。`to_numpy()`の結果は元の表とメモリを共有する場合があるので、独立して変更したいときは `copy=True` を指定します。

#### 例題（演習とは条件が異なります）

In [ ]:
a = df[['temperature','flow']].to_numpy(copy=True)
print('形状:',a.shape,'型:',a.dtype)
print(a[:3,0])
print(a[a[:,1]>0.8][:3])

2列目はflowです。列名は配列に残らないので、列の順番を変えると同じ添字でも別の値になります。

#### よくある間違い

- 列名が配列にも残ると思う
- スライスの終点を含めて数える

### 演習 NP-01

flowとacnをこの順番でNumPy配列にし、先頭4行・両方の列をnp_01に入れてください。shapeは(4, 2)、dtypeは数値型であることを確認します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

まず2列をリストで選び、to_numpy(copy=True)を呼びます。

</details>

<details><summary>ヒント2</summary>

行方向は[:4]で選べます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を np_01 に入れてください。
np_01 = None

In [ ]:
from learning import check_answer
check_answer('NP-01', globals().get('np_01'))

#### 自己確認と振り返り

- shapeのそれぞれの数字は何を数えるか
- 列名が必要な操作ならDataFrameを残した方がよいか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### NP-02　axisとブロードキャストを使う

目安：30分 / 前提：NP-01

#### 学習目標

- 軸を指定して平均を求める
- 列ごとにまとめて単位変換する

2次元配列の平均では、どの方向をまとめるかをaxisで指定します。`axis=0`は行方向をまとめて、各列の平均を返します。`axis=1`は列方向をまとめて、各行の平均を返します。温度と流速のように単位が違う2列の行平均は、計算できても意味がないので避けます。

配列同士の計算では、形状の末尾から見て同じ要素数、または一方が1なら、繰り返して計算するように扱えます。これが**ブロードキャスト**です。(360,2)の配列へ(2,)の倍率を掛けると、全行に列別の倍率が使われます。明示的な繰り返しを書かずに計算できます。

欠損値NaNが含まれる列へ普通のmeanを使うと、結果もNaNになります。`nanmean`は欠損を除いて平均を求めます。計算方法を変えるときは、欠損を無視する判断を説明しましょう。すべて欠損の列の平均はnanmeanでも求められません。

`np.array([60,1000])` はリストから2要素の配列を作る書き方です。

#### 例題（演習とは条件が異なります）

In [ ]:
a = df[['retention_min','flow']].to_numpy()
converted = a * np.array([60,1000])
print('列平均（保持秒・流速µL/min）:',converted.mean(axis=0))
print('温度の欠損除外平均:',np.nanmean(df['temperature'].to_numpy()))

倍率60は分から秒、1000はmLからµLです。axis=0の出力は2要素になり、列順に対応します。

#### よくある間違い

- 異なる単位の行平均を意味のある指標と考える
- axisの番号と出力の形を確認しない

### 演習 NP-02

カンゾウの行だけを選び、保持時間retention_minと流速flowをこの順に配列へ変え、[60, 1000]をブロードキャストして単位変換し、列平均の2要素配列をnp_02に入れてください。axis=1に変えると何が変わるか文章で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

pandasのlocでカンゾウの行と2列を選んでから、配列にします。

</details>

<details><summary>ヒント2</summary>

結果のshapeが(2,)か確認します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を np_02 に入れてください。
np_02 = None

In [ ]:
from learning import check_answer
check_answer('NP-02', globals().get('np_02'))

#### 自己確認と振り返り

- 出力の単位は秒とµL/minになったか
- axis=1がこの例に不適切な理由を説明できるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### NP-03　乱数で測定のばらつきを試す

目安：25分 / 前提：NP-02

#### 学習目標

- 固定シードで再現する
- 乱数の平均と理論値を区別する

乱数は、同じ手順でも値が揺れる状況を試すために使えます。ここでは保持時間に小さなノイズを加え、観測値のばらつきを模擬します。`np.random.default_rng(42)`で乱数を作る生成器を用意します。42はシードという初期値で、同じ初期値から同じ順序で呼び出すと再現できます。

`rng.normal(0, 0.1, size=5)`は平均0、標準偏差0.1の正規分布から5個の値を作ります。標準偏差はばらつきの大きさを表します。5個の標本の平均がぴったり0になるわけではありません。シードを変えると結果が変わり、同じ生成器を続けて使っても次の値へ進みます。

このノイズは実装の練習です。本当の装置の測定誤差が正規分布で、この大きさになるという根拠はありません。条件に合う値を `noise[noise > 0]` のように選ぶ操作も復習しましょう。

#### 例題（演習とは条件が異なります）

In [ ]:
rng = np.random.default_rng(42)
noise = rng.normal(0,0.1,size=5)
print(noise)
print('標本平均:',noise.mean())
print('正の値:',noise[noise>0])

生成する分布の平均は0でも、5個の値の平均は一般に0ではありません。再実行のたび同じ値なのは、セル冒頭で生成器を初期化しているためです。

#### よくある間違い

- 同じ生成器を続けて呼んでも最初の乱数が出ると思う
- 少数の乱数の平均が必ず指定値になると思う

### 演習 NP-03

シード7の生成器で平均0・標準偏差0.2の乱数を10個作り、正の値だけをnp_03に入れてください。全10個の平均が必ず0ではない理由を文章で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

default_rng(7)をセルの先頭で作ります。

</details>

<details><summary>ヒント2</summary>

normalのsize=10、条件は>0です。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を np_03 に入れてください。
np_03 = None

In [ ]:
from learning import check_answer
check_answer('NP-03', globals().get('np_03'))

#### 自己確認と振り返り

- シードを変えず再実行すると同じ結果になるか
- 正の値だけ残すことで分布がどう変わるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

## 3. Matplotlib

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns
from learning import setup_plotting
setup_plotting()


### MPL-01　FigureとAxesで棒グラフを描く

目安：25分 / 前提：pandasのgroupby・NumPyの配列

#### 学習目標

- 図全体と描画領域を区別する
- 集計結果に軸名と単位を付ける

Figureは図全体、Axesはその中の座標軸付きの描画領域です。`fig, ax = plt.subplots()`で2つの値を受け取ります。複数の値を左辺の名前へ順番に入れる書き方です。棒グラフはカテゴリごとの量を比べるときに使います。集計してから描くと、何を比べているかを明確にできます。

棒の高さだけでは単位が分かりません。`set_xlabel()`と`set_ylabel()`で軸名と単位を入れます。`set_title()`は図の見出しです。最後の`plt.show()`で図を表示します。棒はゼロから始めると長さの比較を誤解しにくくなります。

この例の平均保持時間は異なる成分と条件が混在した結果です。図から差は見えますが、生薬だけの影響とは限りません。

例題の `ax.set(xlabel=..., ylabel=..., title=...)` は軸名とタイトルをまとめて設定する書き方です。

#### 例題（演習とは条件が異なります）

In [ ]:
summary = df.groupby('herb')['retention_min'].mean()
fig, ax = plt.subplots()
ax.bar(summary.index,summary.values)
ax.set(xlabel='生薬',ylabel='平均保持時間（分）',title='生薬別の平均保持時間')
plt.show()

棒の高さは生薬別の平均保持時間です。検体数や条件の違いを踏まえて読みます。

#### よくある間違い

- 360行を集計せず同じカテゴリへ重ねて描く
- 軸の単位を書かない

### 演習 MPL-01

装置instrument別の測定件数を棒グラフにしてください。横軸に装置、縦軸に測定件数（件）、タイトルを付け、Figureをmpl_01に入れてください。2本の棒の合計が360件であることを確認します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

groupbyのsize()は欠損に関係なく各組の行数を数えます。

</details>

<details><summary>ヒント2</summary>

fig, axのfigをmpl_01として受け取れます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を mpl_01 に入れてください。
mpl_01 = None

In [ ]:
from learning import check_answer
check_answer('MPL-01', globals().get('mpl_01'))

#### 自己確認と振り返り

- 棒の合計と元の行数が一致するか
- FigureとAxesのどちらへbarを呼んだか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### MPL-02　目的に合わせて図を選ぶ

目安：35分 / 前提：MPL-01

#### 学習目標

- 散布図とヒストグラムを使い分ける
- 順序を持つ集計に折れ線を使う

2つの数値の関係を見る散布図は `scatter()`、1つの数値の分布を見るヒストグラムは `hist()`を使います。ヒストグラムのbinsは区間数で、細かくしすぎると偶然のばらつきが目立ちます。区間数を変えても残る特徴を探しましょう。

折れ線の `plot()`は点を順に結びます。このCSVの行順には時間の意味がないため、行番号順に保持時間を結んでも時間変化とは言えません。例題では流速を0.1刻みの区分へ丸め、区分ごとの平均を流速順に描きます。round(1)は小数1桁への丸めです。

散布図で傾向が見えても、成分やほかの測定条件が影響している可能性があります。軸の範囲と単位を確認し、図の形から言えることと言えないことを分けます。

`assign(flow_bin=...)` は元の表を変えず、区分の列を加えた新しい表を返します。

#### 例題（演習とは条件が異なります）

In [ ]:
by_flow = df.assign(flow_bin=df['flow'].round(1)).groupby('flow_bin')['retention_min'].mean()
fig, ax = plt.subplots()
ax.plot(by_flow.index,by_flow.values,marker='o',label='流速区分の平均')
ax.set(xlabel='流速区分（mL/min）',ylabel='平均保持時間（分）')
ax.legend()
plt.show()
fig, ax = plt.subplots()
ax.scatter(df['acn'],df['retention_min'],alpha=0.5)
ax.set(xlabel='ACN（%）',ylabel='保持時間（分）')
plt.show()
fig, ax = plt.subplots()
ax.hist(df['resolution'],bins=10,edgecolor='white')
ax.set(xlabel='分離度Rs',ylabel='件数')
plt.show()

折れ線は並べた流速区分の平均を結んでいます。散布図では同じACN付近でも保持時間が広がり、ACNだけで値が決まらないことが見えます。 最後のヒストグラムは分離度の分布で、棒の高さが各区間の件数です。

#### よくある間違い

- 行順の折れ線を測定日の推移と思う
- ヒストグラムの縦軸を測定値と思う

### 演習 MPL-02

保持時間の分布を12区間のヒストグラムで描き、Figureをmpl_02に入れてください。横軸を保持時間（分）、縦軸を件数にし、棒の件数の合計と行数を比べてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

hist(df[列名], bins=12)を使います。

</details>

<details><summary>ヒント2</summary>

すべての保持時間が記録されているので合計は360です。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を mpl_02 に入れてください。
mpl_02 = None

In [ ]:
from learning import check_answer
check_answer('MPL-02', globals().get('mpl_02'))

#### 自己確認と振り返り

- binsを6や24に変えると印象は変わるか
- データの順序に意味がある図とない図を区別する

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### MPL-03　複数の図を並べて保存する

目安：30分 / 前提：MPL-02

#### 学習目標

- subplotsで図を並べる
- 凡例と日本語ラベルを付ける
- PNGを保存する

`plt.subplots(1,2)`は1行2列の描画領域を作り、axesという配列で返します。`axes[0]`と`axes[1]`へ別の図を描くと、同じ紙面で比較できます。`fig.tight_layout()`は軸名などの重なりを軽減し、`fig.savefig()`は図全体を画像へ保存します。dpiは1インチ当たりの画素数です。

複数系列の凡例には、作図時のlabelと`legend()`を使います。同じ処理を繰り返す場合の `for name in リスト:` は、値を一つずつ取り出して字下げした処理を実行する構文です。例題では2装置を同じAxesへ描きます。色に加えて凡例名や点の形でも識別できると読みやすくなります。

日本語フォントは章の準備で同梱ファイルを登録済みです。画像を開き、軸名が欠けていないかを確かめます。画面に図が出たことと、保存先へ正しい図を書けたことは別に確認しましょう。

#### 例題（演習とは条件が異なります）

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
for name in ['LC-A','LC-B']:
    part = df[df['instrument']==name]
    axes[0].scatter(part['flow'],part['retention_min'],label=name,alpha=0.5)
axes[0].set(xlabel='流速（mL/min）',ylabel='保持時間（分）')
axes[0].legend()
axes[1].hist(df['resolution'],bins=12)
axes[1].set(xlabel='分離度Rs',ylabel='件数')
fig.tight_layout()
fig.savefig('outputs/overview.png',dpi=150)
plt.show()

左は流速と保持時間の関係、右は分離度の分布です。両者の軸は別の量なので、縦の位置をそのまま対応付けません。

#### よくある間違い

- plt.savefigの時点で別のFigureを選んでしまう
- 画面表示だけを見て保存画像の欠けを確認しない

### 演習 MPL-03

1行2列の図を作り、左にACNと保持時間の散布図、右に保持時間のヒストグラムを描いてください。軸名・単位を付け、Figureをmpl_03に入れ、outputs/my_overview.pngへ保存します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

mpl_03, axes = plt.subplots(1,2,...)とします。

</details>

<details><summary>ヒント2</summary>

各axesに描いてtight_layout後、mpl_03.savefigを使います。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を mpl_03 に入れてください。
mpl_03 = None

In [ ]:
from learning import check_answer
check_answer('MPL-03', globals().get('mpl_03'))

#### 自己確認と振り返り

- 保存した画像を開きラベルを読めるか
- 2つの図で答えられる問いの違いは何か

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

## 4. seaborn

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns
from learning import setup_plotting
setup_plotting()


### SNS-01　DataFrameから分布を描く

目安：25分 / 前提：MatplotlibのFigureとAxes

#### 学習目標

- 列名で作図する
- hueでグループを分ける

seabornはDataFrameの列名を指定して統計的な図を描くライブラリです。`data=df, x='retention_min'`のように書くと、列を取り出す処理を繰り返さずに使えます。`hue`は色を分ける分類列の指定で、凡例も作られます。

`histplot()`はヒストグラムを描きます。全体の分布と、生薬ごとに分けた分布を比べると、どのグループが全体の形に関わるかを見られます。ただし件数が違うグループを比較すると、棒の高さには測定件数の差も現れます。

seabornもMatplotlibのAxesに描画します。`ax=ax`で描画先を渡し、その後 `ax.set()`で日本語の軸名を設定できます。テーマを設定するとフォントも変わる場合があるため、`sns.set_theme()`の後に `setup_plotting()`を呼び直します。

#### 例題（演習とは条件が異なります）

In [ ]:
sns.set_theme(style='whitegrid')
setup_plotting()
fig, ax = plt.subplots()
sns.histplot(data=df,x='retention_min',hue='herb',bins=18,element='step',ax=ax)
ax.set(xlabel='保持時間（分）',ylabel='件数')
plt.show()

成分ごとの分布が重なっており、保持時間だけでは成分名は決まりません。色は凡例と合わせて読みます。

#### よくある間違い

- hueへ数値列を渡したときの意味を確認しない
- seabornをMatplotlibと独立した描画環境と思う

### 演習 SNS-01

分離度resolutionのヒストグラムを装置instrumentで色分けし、Figureをsns_01へ入れてください。軸名を分離度Rs・件数とし、凡例で装置が識別できることを確認します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

histplotへdata、x、hue、axを渡します。

</details>

<details><summary>ヒント2</summary>

色分けした分布は重なるためelement="step"も使えます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を sns_01 に入れてください。
sns_01 = None

In [ ]:
from learning import check_answer
check_answer('SNS-01', globals().get('sns_01'))

#### 自己確認と振り返り

- 凡例にLC-AとLC-Bがあるか
- 件数の差が棒の高さへ影響していないか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### SNS-02　カテゴリ比較と変数間の関係

目安：30分 / 前提：SNS-01

#### 学習目標

- 箱ひげ図を読む
- 色分け散布図で混在を確かめる

箱ひげ図の箱は第1四分位点から第3四分位点まで、箱の中の線は中央値です。seabornの既定のひげは箱の幅の1.5倍の範囲内にある端の値まで伸び、その外の点を個別に描きます。外側の点が自動的に誤測定を意味するわけではありません。

`boxplot()`はカテゴリごとの分布比較、`scatterplot()`は2つの数値の関係に向いています。散布図のhueに生薬名を指定すると、全体の傾向がカテゴリの混在によって見えていないかを点検できます。styleも指定すると点の形を変えられ、色だけに頼らず区別できます。

同じACNでも保持時間に幅があるのは、ほかの条件も変わっているためです。図で関係を見つけた後に、その原因を確かめるには、条件をそろえた実験など別の検討が必要です。

#### 例題（演習とは条件が異なります）

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
sns.boxplot(data=df,x='herb',y='retention_min',ax=axes[0])
axes[0].set(xlabel='生薬',ylabel='保持時間（分）')
sns.scatterplot(data=df,x='acn',y='retention_min',hue='herb',style='herb',ax=axes[1])
axes[1].set(xlabel='ACN（%）',ylabel='保持時間（分）')
fig.tight_layout()
plt.show()

箱の中央値と広がりは別の情報です。散布図の色分けによって同じ生薬内の傾向も確かめます。

#### よくある間違い

- 箱の外の点を確認なしに削除する
- 異なる成分量の大小を薬効の大小と読む

### 演習 SNS-02

流速flowと保持時間retention_minの散布図を、生薬herbで色と形を分けて描き、Figureをsns_02に入れてください。「流速だけでは保持時間を説明しきれない」根拠を文章セルに書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

scatterplotのhueとstyleに同じherb列を渡せます。

</details>

<details><summary>ヒント2</summary>

同じ流速付近にある点の縦の広がりを見ます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を sns_02 に入れてください。
sns_02 = None

In [ ]:
from learning import check_answer
check_answer('SNS-02', globals().get('sns_02'))

#### 自己確認と振り返り

- 凡例と点の形で生薬を区別できるか
- 観察された傾向と原因の断定を分けたか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### SNS-03　相関を一覧にして解釈する

目安：30分 / 前提：SNS-02・pandasの列選択

#### 学習目標

- 相関係数を読み取る
- ヒートマップを整える
- 作図ライブラリを使い分ける

相関係数は2つの数値の直線的な関係の強さと向きを、-1から1で表します。正なら一方が大きいと他方も大きい傾向、負なら逆方向です。0に近くても曲線的な関係がないとは限りません。

`corr()`で数値列どうしの相関行列を作り、`heatmap()`で値を色に変換します。対角線は同じ変数なので1です。`annot=True`で数値も表示すると色だけに頼らず読めます。`vmin=-1,vmax=1,center=0`で色の範囲を固定し、別の図とも比較しやすくします。

欠損がある場合、pandasは列の組ごとに両方の値が記録された行を使います。すべての係数が同じ行集合から計算されるとは限りません。相関が強くても因果関係や測定前の予測に使えることを保証しません。seabornで統計的な図を作り、Matplotlibで軸名や保存を調整すると役割を分担できます。

#### 例題（演習とは条件が異なります）

In [ ]:
corr = df[['temperature','flow','acn','retention_min']].corr()
fig, ax = plt.subplots()
sns.heatmap(corr,annot=True,fmt='.2f',vmin=-1,vmax=1,center=0,cmap='RdBu_r',ax=ax)
ax.set_title('測定条件と保持時間の相関')
plt.tight_layout()
plt.show()

流速と保持時間の負の相関などを、数値の符号と大きさで読みます。温度に欠損があるため係数ごとに使用件数が違います。

#### よくある間違い

- 相関0をあらゆる関係がない証拠とする
- ピーク面積と成分量の相関を測定前の予測へ無条件に使う

### 演習 SNS-03

図を参考に「負の相関の意味」「相関から原因を決められない理由」「欠損によって使用行が変わること」を含む日本語60字以上の説明をsns_03に入れてください。自動確認は文字数だけで、内容は振り返りで確認します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

流速と保持時間の関係を具体例にします。

</details>

<details><summary>ヒント2</summary>

ほかの測定条件も変わっていることを理由に加えます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を sns_03 に入れてください。
sns_03 = None

In [ ]:
from learning import check_answer
check_answer('SNS-03', globals().get('sns_03'))

#### 自己確認と振り返り

- 負の相関を具体的な変数名で説明したか
- 原因の断定と欠損への言及を自分で確認したか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

## 5. scikit-learn

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns
from learning import setup_plotting
setup_plotting()
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, recall_score, confusion_matrix


### ML-01　予測に使える情報を選んで分割する

目安：30分 / 前提：pandas・NumPy・グラフの基礎

#### 学習目標

- 特徴量と目的変数を分ける
- 訓練・テストへ先に分割する

保持時間を測定前に予測する場面を考えます。予測の入力が特徴量X、予測したい答えが目的変数yです。連続した数値を予測する課題を回帰、区分を予測する課題を分類と呼びます。

入力には、測定前に分かる温度・流速・ACN比率・成分名・装置を使います。ピーク面積や測定済み分離度は測定後の情報です。保持時間そのものを入力すれば答えを見ていることになり、未知の測定へ使えません。

`train_test_split`はXとyの対応を保ったまま行を分け、訓練用とテスト用の4つを返します。test_size=0.2は20%をテストへ残す指定です。random_stateは分割を再現する初期値です。モデルも欠損補完も訓練データで学習し、テストは最後の評価用に保留します。同一検体の反復なら検体単位で分ける必要がありますが、この教材では各行が独立した架空検体です。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('訓練:',X_train.shape,'テスト:',X_test.shape)
print('行の重複:',len(X_train.index.intersection(X_test.index)))

360行のうち288行が訓練、72行がテストです。列数は5で、同じ行が両方に入っていません。

#### よくある間違い

- 全件で補完・標準化してから分ける
- 測定後の情報を測定前の予測に使う

### 演習 ML-01

同じ5つの特徴量から保持時間を予測するため、test_size=0.25、random_state=7で分割してください。ml_01にはX_train、X_test、y_train、y_testというキーの辞書で4つを入れ、テストが90行で重複がないことを確かめます。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

Xとyを同時にtrain_test_splitへ渡します。

</details>

<details><summary>ヒント2</summary>

返された4つを同じ名前のキーへ入れます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_01 に入れてください。
ml_01 = None

In [ ]:
from learning import check_answer
check_answer('ML-01', globals().get('ml_01'))

#### 自己確認と振り返り

- 各入力列は予測する時点で分かるか
- 同じ検体が訓練とテストに混ざる状況を説明できるか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ML-02　前処理とモデルをPipelineにつなぐ

目安：40分 / 前提：ML-01

#### 学習目標

- 補完・標準化・カテゴリ変換を使う
- 訓練だけでfitする

欠損を中央値で埋めるSimpleImputer、各数値列の平均とばらつきをそろえるStandardScaler、文字カテゴリを0/1の列へ変えるOneHotEncoderを組み合わせます。未知のカテゴリにはhandle_unknown='ignore'を指定して停止を避けますが、未学習カテゴリを正確に予測できる保証ではありません。

ColumnTransformerは列の種類ごとに処理を分けます。Pipelineは処理を順につなぐ仕組みです。リストの各要素は `(処理名, 処理のオブジェクト)` という2つ組のタプルです。Ridgeは係数が極端に大きくならないよう抑えた線形回帰で、alphaがその強さです。

分割後に `model.fit(X_train,y_train)` を呼ぶと、補完値・標準化の基準・回帰係数を訓練だけで学びます。`predict(X_test)`は学んだ前処理をそのまま使います。テストへfitを呼びません。後の交差検証でもPipeline全体を渡せば、各分割の訓練部分だけで前処理を学習できます。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
preprocess = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
model = Pipeline([('prepare', preprocess), ('predict', Ridge(alpha=1.0))])
model.fit(X_train,y_train)
print('先頭3件の予測（分）:',model.predict(X_test.iloc[:3]))

補完・文字の変換を自分でテストへ繰り返す必要はありません。モデルが保持している訓練時の設定が使われます。

#### よくある間違い

- テストへfit_transformを呼ぶ
- カテゴリ列を単純な連番へ変えて大小関係を作る

### 演習 ML-02

このレッスンの例題と同じtest_size=0.2、random_state=42で分割し、Ridgeのalphaを10.0にしたPipelineを組み、訓練データにfitしてください。学習済みPipelineをml_02に入れ、テスト先頭5行への予測を表示します。前処理に使った中央値が訓練だけから求まっているか確認します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

ColumnTransformerとPipelineの構成は例題を参考にし、alphaを変更します。

</details>

<details><summary>ヒント2</summary>

ml_02.fitへ渡すのはX_trainとy_trainだけです。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_02 に入れてください。
ml_02 = None

In [ ]:
from learning import check_answer
check_answer('ML-02', globals().get('ml_02'))

#### 自己確認と振り返り

- fitへテストを渡していないか
- predictで欠損とカテゴリを処理できたか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ML-03　単純な予測と比べて評価する

目安：30分 / 前提：ML-02

#### 学習目標

- ベースラインと比較する
- MAEとR²を読む

機械学習が役に立つかは、単純な予測と比べて確かめます。DummyRegressorはここでは訓練の保持時間の平均を、すべての測定へ返します。これが比較基準となるベースラインです。入力条件を使わなくても得られる予測より改善しているかを見ます。

MAEは予測と実際の差の絶対値を平均したものです。保持時間なら単位は分で、小さいほど誤差が小さくなります。R²は平均予測に対する改善を表す指標で、1が完全一致、0付近がテスト内の平均を返す程度です。負にもなり、その場合はその平均より悪い予測です。訓練平均を返すDummyのR²は必ず0ではありません。

例題は決めたRidgeを1回評価する学習用の実験です。テスト結果を見て条件を繰り返し変えると、テストも選択へ使ってしまいます。モデルの比較・選択は後の交差検証で行い、最後のテストは保留します。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
preprocess = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
model = Pipeline([('prepare', preprocess), ('predict', Ridge(alpha=1.0))])
model.fit(X_train,y_train)
pred = model.predict(X_test)
baseline = DummyRegressor(strategy='mean').fit(X_train,y_train)
base_pred = baseline.predict(X_test)
print('Ridge MAE（分）:',mean_absolute_error(y_test,pred))
print('平均予測 MAE（分）:',mean_absolute_error(y_test,base_pred))
print('R²:',r2_score(y_test,pred))

MAEをベースラインと同じテスト行で比べます。R²とMAEは異なる尺度なので、値の大小を直接比較しません。

#### よくある間違い

- R²を正解率と呼ぶ
- ベースラインだけ別のテストで評価する

### 演習 ML-03

test_size=0.25、random_state=42、Ridge alpha=1.0で、同じテスト行に対するRidgeと平均予測を比較してください。ml_03にmae、baseline_mae、r2の3キーで数値を入れます。MAEの単位とベースラインより改善したかを文章で説明してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

分割以降の前処理は訓練だけで学習します。

</details>

<details><summary>ヒント2</summary>

mean_absolute_errorへ実測値と予測値を渡します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_03 に入れてください。
ml_03 = None

In [ ]:
from learning import check_answer
check_answer('ML-03', globals().get('ml_03'))

#### 自己確認と振り返り

- MAEを分という単位で説明したか
- 同じテスト行で2つのモデルを比較したか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ML-04　分類の正解率と見逃しを読む

目安：35分 / 前提：ML-03

#### 学習目標

- 回帰と分類を区別する
- 混同行列とrecallを読む

分離度が1.5以上かという学習用の区分を作ります。`(df['resolution'] >= 1.5).astype(int)`でTrueを1、Falseを0へ変えます。これは二値分類の練習であり、実試料の規格適合や測定法の妥当性を判定するものではありません。

LogisticRegressionは名前にRegressionがありますが、ここでは分類に使います。stratify=yは分割後も0と1の割合を近く保つ指定です。DummyClassifierは訓練で多い方のクラスを常に予測するベースラインにします。

accuracyは全体の正解割合です。recallは実際に1だった行のうち、1と予測できた割合です。この定義では見逃しは「実際は分離度区分1なのに0と予測する」ことです。混同行列は行が実際、列が予測で、labels=[0,1]なら左上が0を0、右下が1を1と予測した件数です。クラスの意味を逆にするとrecallの意味も変わるので、値だけを読まないでください。

`zero_division=0`は、実際のクラス1が0件で再現率を計算できない場合に0を返す指定です。その場合の0は「全件見逃した」と同じ意味ではなく、評価対象の1がいないことを併せて報告します。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = (df['resolution'] >= 1.5).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocess = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
model = Pipeline([('prepare', preprocess), ('predict', LogisticRegression(max_iter=1000))])
model.fit(X_train,y_train)
pred = model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
base_pred = baseline.predict(X_test)
print('正解率:',accuracy_score(y_test,pred))
print('クラス1の再現率:',recall_score(y_test,pred,zero_division=0))
print('混同行列：行=実際、列=予測、順序0,1')
print(confusion_matrix(y_test,pred,labels=[0,1]))

左下は実際1・予測0の件数です。精度の評価では正解率だけでなく、どちらの取り違えが起きたかを読みます。

#### よくある間違い

- クラス1の意味を決めずrecallを読む
- 正解率が高ければ両方のクラスで良いと考える

### 演習 ML-04

同じ入力列を使い、test_size=0.25、random_state=42、stratify=yで分類してください。ml_04にaccuracy、baseline_accuracy、recall、matrix（labels=[0,1]の混同行列）の4キーを入れ、取り違えの方向を説明します。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

LogisticRegressionのPipelineとDummyClassifierを同じ訓練行で学習します。

</details>

<details><summary>ヒント2</summary>

confusion_matrixの行が実際、列が予測です。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_04 に入れてください。
ml_04 = None

In [ ]:
from learning import check_answer
check_answer('ML-04', globals().get('ml_04'))

#### 自己確認と振り返り

- 混同行列の合計が90か
- どのクラスのrecallを計算したかを言葉にしたか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ML-05　訓練に合いすぎるモデルを見つける

目安：35分 / 前提：ML-04

#### 学習目標

- 訓練誤差と検証誤差を比べる
- 最終テストを選択に使わない

訓練データへ細かく合わせすぎて、未知のデータで誤差が大きくなる状態を過学習と呼びます。決定木は条件を枝分かれさせて予測するモデルです。max_depthは枝分かれの深さの上限で、Noneなら深さの制限を置きません。

ここでは最初に確保したテストを使わず、訓練データの中をさらに学習用と検証用へ分けます。深さ2と無制限の木を同じ分割で比較し、訓練MAEと検証MAEを並べます。訓練だけ極端に良くても、モデル選択の根拠にはなりません。

`for depth in [2,None]:`は深さの候補を一つずつ取り出す繰り返しです。`rows=[]`で空のリストを作り、appendで結果の辞書を追加して最後に表へ変えます。各候補でPipelineを新しく作り、補完の基準も学習側だけから学びます。有限の検証データなので、たった1回の分割でモデルの優劣が固定されるわけではありません。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train,y_train,test_size=0.25,random_state=7)
rows = []
for depth in [2,None]:
    preprocess = ColumnTransformer([
        ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
    ])
    model = Pipeline([('prepare', preprocess), ('predict', DecisionTreeRegressor(max_depth=depth,random_state=42))])
    model.fit(X_fit,y_fit)
    rows.append({'depth':str(depth),'train_mae':mean_absolute_error(y_fit,model.predict(X_fit)),
                 'valid_mae':mean_absolute_error(y_valid,model.predict(X_valid))})
comparison = pd.DataFrame(rows)
display(comparison)

無制限の木は訓練誤差をほぼゼロにできても、検証誤差は残ります。必ず浅い木より悪くなるわけではない点も表で確かめます。

#### よくある間違い

- 訓練誤差が最小のモデルを必ず採用する
- 深さを変えるたびテストを見て最終評価に使う

### 演習 ML-05

深さ3とNoneの決定木を、最初にtest_size=0.2・random_state=42でテストを保留し、残りをtest_size=0.25・random_state=7で学習用と検証用に分けて比較してください。ml_05をdepth・train_mae・valid_maeの2行の表にし、どの誤差を選択の参考にするか文章で説明します。最終テストは使いません。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

例題の候補リストを[3,None]へ変更します。

</details>

<details><summary>ヒント2</summary>

検証用はX_trainから分け、X_testは参照しません。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_05 に入れてください。
ml_05 = None

In [ ]:
from learning import check_answer
check_answer('ML-05', globals().get('ml_05'))

#### 自己確認と振り返り

- 最終テストをこの比較で使っていないか
- 無制限の木の訓練誤差と検証誤差の差を説明したか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ML-06　交差検証をしてから最後のテストへ

目安：40分 / 前提：ML-05

#### 学習目標

- Pipelineを交差検証する
- CVと最終テストの役割を区別する

交差検証は、訓練データを複数の組に分け、1組ずつ検証に回す方法です。cv=5では5通りを評価します。各回で前処理を学び直すため、配列へ一括で補完したものではなくPipeline全体を渡します。cross_val_scoreは大きいほど良い評価値を返す設計なので、MAEは負の値として返ります。先頭のマイナス記号で正の誤差へ戻します。

候補のalphaを訓練内のCV平均で比較し、選んだ設定を全訓練データにfitします。その後に初めて、保留したテストで評価します。CVの平均と標準偏差は分割による変動の目安で、将来の誤差を保証する区間ではありません。

結果を再現するために、分割、特徴量、前処理、モデル設定、指標を記録します。教材で何度も試すとテストに慣れてしまう点にも注意してください。実際の最終評価では、本当に選択へ使っていない測定が必要です。

例題ではalpha=1の1候補だけを評価します。演習では3候補を比較してから最終テストへ進みます。

`scoring='neg_mean_absolute_error'`は評価指標として負のMAEを選ぶ指定です。既定の指標に任せず、比較したい量を明示します。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
preprocess = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
model = Pipeline([('prepare', preprocess), ('predict', Ridge(alpha=1.0))])
scores = -cross_val_score(model,X_train,y_train,cv=5,scoring='neg_mean_absolute_error')
print('各分割のMAE（分）:',scores)
print('平均:',scores.mean(),'標準偏差:',scores.std())
model.fit(X_train,y_train)
print('選択後の最終テストMAE（分）:',mean_absolute_error(y_test,model.predict(X_test)))


5つのCV誤差は少しずつ異なります。最終テストMAEがCV平均と完全に一致しないのは、別の測定を評価しているためです。

#### よくある間違い

- 負のMAEをそのまま小さいほど良いと読む
- テストで選んだalphaをテストで最終評価する

### 演習 ML-06

最初にtest_size=0.2、random_state=42で分割します。alpha候補0.1、1.0、10.0を訓練データだけの5分割CVで比較してください。CV平均MAEが最小のalphaを選び、全訓練で再学習し最後にテストを1度評価します。ml_06はcv（alpha・maeの3行表）、best_alpha、test_maeの辞書にします。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

候補ごとにPipelineを作り、cross_val_scoreへX_train,y_trainだけを渡します。

</details>

<details><summary>ヒント2</summary>

mae列が最小の行はidxmin()で取り出せます。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を ml_06 に入れてください。
ml_06 = None

In [ ]:
from learning import check_answer
check_answer('ML-06', globals().get('ml_06'))

#### 自己確認と振り返り

- alpha選択と最終評価に別のデータを使ったか
- CVのばらつきを保証と誤解していないか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

## 6. 総合演習

この章の準備セルを最初に実行してください。前の章の回答は使いません。例題は実行して変更する見本、演習は自分で書く問題です。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from learning import check_answer
df = pd.read_csv('data/hplc.csv')
analytes = pd.read_csv('data/analytes.csv')
Path('outputs').mkdir(exist_ok=True)
import matplotlib.pyplot as plt
import seaborn as sns
from learning import setup_plotting
setup_plotting()
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, recall_score, confusion_matrix


### ALL-01　測定表を整えて集計を残す

目安：35分 / 前提：pandasからscikit-learnまで

#### 学習目標

- 加工前後の行数を記録する
- 結合と集計を組み合わせる

ここからは道具ごとの練習をつなぎます。最初に「成分量が記録された測定を生薬別に要約する」と目的を決めます。使う列と欠損の扱いを先に決めると、必要以上の行を落とさずに済みます。

読み込み、欠損数の確認、必要な行の選択、マスタ結合、集計、保存の順に進みます。中間の表の名前を分けると、どこで件数が変わったかを追えます。平均と有効件数を並べ、保存後には列名と値を読み直します。

実データでは測定単位や抽出条件がそろっているかも重要です。架空データで動いた処理をそのまま実測へ適用せず、1行が何を表すか、欠損がなぜ生じたか、マスタが一意かを確認してから進めます。

#### 例題（演習とは条件が異なります）

In [ ]:
raw = pd.read_csv('data/hplc.csv')
print('元の行数:',len(raw))
clean = raw.dropna(subset=['content_mg_g']).copy()
print('残った行数:',len(clean))
summary = clean.groupby('herb')['content_mg_g'].agg(['count','mean'])
display(summary)

除外した行数と生薬別の件数を見ます。成分量の平均だけ保存するより、どれだけの測定を使ったかが分かります。

#### よくある間違い

- 必要のない列の欠損まで理由なく除外する
- 保存した表から単位や処理内容が分からない

### 演習 ALL-01

CSVを新たに読み、成分量が欠損した行を除き、準備セルのanalytes（data/analytes.csvの成分マスタ）をmany_to_oneで結合してください。成分類class別のcontent_mg_gのcountとmeanを表all_01に入れ、outputs/final_summary.csvへ分類名を通常列として保存します。除外行数も表示してください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

成分量の欠損だけを除外します。

</details>

<details><summary>ヒント2</summary>

groupby後にagg(["count","mean"])を使います。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を all_01 に入れてください。
all_01 = None

In [ ]:
from learning import check_answer
check_answer('ALL-01', globals().get('all_01'))

#### 自己確認と振り返り

- 入力CSVを上書きしていないか
- 行数・欠損・結合キー・単位を記録したか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ALL-02　図と文章で結果を伝える

目安：35分 / 前提：ALL-01（回答済みでなくてもよい）

#### 学習目標

- 問いに合う図を選ぶ
- 根拠と限界を文章にする

グラフを作る前に、答えたい問いを1つに絞ります。「流速と保持時間にどのような関係があるか」なら散布図、「成分ごとの広がりはどう違うか」なら箱ひげ図が向いています。見た目を整えることに加え、どの行を使った図かを説明します。

説明文は、見えた傾向、図のどこが根拠か、ここからは分からないことの順で書くと伝わります。例えば「同じ流速でも縦に点が広がる」という観察は図で確認できます。一方、「装置が原因だ」はこの図だけでは確認できません。

今回の図は架空の生成式の結果です。実測の機構を証明するものではありません。色分け・形・軸の単位を整え、図だけを受け取った人にも比較対象が分かるようにします。

#### 例題（演習とは条件が異なります）

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df,x='herb',y='retention_min',ax=ax)
ax.set(xlabel='生薬',ylabel='保持時間（分）',title='架空測定の保持時間の分布')
fig.tight_layout()
plt.show()

中央値と箱の広がりを分けて読みます。異なる条件を混ぜた分布なので、成分の違いだけの効果ではありません。

#### よくある間違い

- 図の装飾に集中し問いが不明になる
- 観察事実と仮説を同じ強さで断定する

### 演習 ALL-02

準備セルのdf（欠損除外前の360行）を使います。ACNと保持時間の散布図を生薬別に色・形で分け、outputs/final_plot.pngへ保存してください。all_02をfigureとreportの辞書にし、reportに「傾向・根拠・限界」を含む60字以上の説明を書いてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

scatterplotのhueとstyleをherbにします。

</details>

<details><summary>ヒント2</summary>

図を見て、同じACNでのばらつきにも言及します。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を all_02 に入れてください。
all_02 = None

In [ ]:
from learning import check_answer
check_answer('ALL-02', globals().get('all_02'))

#### 自己確認と振り返り

- 文章の根拠を図上で示せるか
- 実試料への一般化や因果を断定していないか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。

### ALL-03　予測の結果と限界を報告する

目安：40分 / 前提：ML-06・ALL-02（回答済みでなくてもよい）

#### 学習目標

- 一連の分析を再現する
- ベースライン・CV・テストを報告する
- 実測へ移す前の課題を書く

最後は保持時間の予測を、読み込みから報告までつなぎます。測定前に分かる5列だけを選び、20%を最終テストに残します。中央値補完・標準化・カテゴリ変換をPipelineへ入れ、訓練だけで5分割CVをします。モデル設定はここではalpha=1.0に固定し、CVでテスト結果を選びません。

CVで大きな不具合がないか確認してから、訓練全体で学習し、最終テストを1度だけ評価します。同じテスト行へ訓練平均のベースラインも出し、MAEを比較します。図には実際の保持時間と予測値を描き、同じなら乗る対角線も加えます。

報告には入力、分割、指標と単位、ベースラインとの差、限界を含めます。今回は架空の生成式を学ぶ結果です。実測へ進むなら検体・ロットの重複、装置や日付の違い、将来のデータでの評価を考える必要があります。

例題は予測値と架空の実測値の図を作ります。演習はCV・数値評価・報告をまとめます。固定したモデル設定で評価手順を一通り実行する練習です。

解答の文章はf文字列で数値を埋め込みます。例えば `f'MAEは{mae:.3f}分'` は変数maeを小数3桁で表示します。報告は手で数値を書いても構いません。

#### 例題（演習とは条件が異なります）

In [ ]:
numeric = ['temperature', 'flow', 'acn']
categorical = ['analyte', 'instrument']
X = df[numeric + categorical]
y = df['retention_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
preprocess = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
model = Pipeline([('prepare', preprocess), ('predict', Ridge(alpha=1.0))])
model.fit(X_train,y_train)
pred = model.predict(X_test)
baseline = DummyRegressor(strategy='mean').fit(X_train,y_train)
base_pred = baseline.predict(X_test)
fig, ax = plt.subplots()
ax.scatter(y_test,pred,alpha=0.6)
limits = [min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]
ax.plot(limits,limits,linestyle='--',color='black',label='予測=実測')
ax.set(xlabel='架空の実測値（分）',ylabel='予測値（分）')
ax.legend()
plt.show()

対角線に近いほど誤差が小さく、上側では過大予測、下側では過小予測です。誤差の大きさが保持時間によって変わらないかも確認します。

#### よくある間違い

- CVとテストの値を混ぜて一つの精度と呼ぶ
- 架空データの良い性能を実試料での保証とする

### 演習 ALL-03

CSVを新たに読み、測定前の5列から保持時間を予測してください。20%をrandom_state=42で保留し、Ridge alpha=1のPipelineを訓練だけの5分割CVで評価します。全訓練で学習後、最終テストと訓練平均ベースラインのMAEを求めます。all_03にcv_mae（5値）、test_mae、baseline_mae、report（入力・分割・単位・比較・限界を含む100字以上）を入れてください。

使用データ：`data/hplc.csv`（架空測定データ）、結合には `data/analytes.csv`。この章の準備セルを先に実行します。ほかの演習への依存はありません。

<details><summary>ヒント1</summary>

前処理を先に全データへ適用しないでください。

</details>

<details><summary>ヒント2</summary>

CVにはX_trainだけ、最後のpredictにはX_testを使います。

</details>

In [ ]:
# TODO：問題文に沿ってコードを書き、結果を all_03 に入れてください。
all_03 = None

In [ ]:
from learning import check_answer
check_answer('ALL-03', globals().get('all_03'))

#### 自己確認と振り返り

- 前処理のfitへテストが混ざっていないか
- 報告に単位・比較・限界を含めたか
- 自分の仕事に適用する前に何を追加検証するか

グラフは軸・単位・比較対象・凡例、文章は根拠と限界を自分でも点検します。自動確認だけで解釈の正しさは判定できません。